# HIF sensor (Phase 0 + Phase 1 + Phase 1.5): simulate HIF, test VeHIF 22 kV, adapt & verify C firmware

Bấm **Run All** trên Kaggle (CPU là đủ, không cần Internet). Để chạy Phase 1 & Phase 1.5, gắn dataset `dougpsg/highimpedance-vegetation-fault-data-set` (106 GB sẵn trên Kaggle, không tải về máy).

All simulated arcs use the Emanuel model; VeHIF currents are real 22 kV 50 Hz vegetation-fault recordings passed through a simulated sensor and load.

In [ ]:
!mkdir -p hif results/figures firmware build

In [ ]:
%%writefile hif/__init__.py
"""High-impedance fault (HIF) sensor: front-end model, waveform simulator, features."""


In [ ]:
%%writefile hif/frontend.py
"""Analog front end + ESP32-S3 ADC model. Component values match circuit/front_end_ac.cir.

Supports both 60 Hz (US 12.47 kV, FS_ADC = 15,360 Hz) and 50 Hz (Vietnam / Australia 22 kV,
FS_ADC = 12,800 Hz) grid operation on the exact same physical 4th-order Butterworth Sallen-Key
anti-aliasing hardware (fc ≈ 2,045 Hz), as well as an optional resistor-retuned 50 Hz variant.
"""
import numpy as np
from scipy import signal

F_LINE = 60.0
F_LINE_VN = 50.0                    # Vietnam / Australia / IEC 50 Hz grid frequency
N_CYCLE = 256                       # ADC samples per power cycle (same for 50 Hz and 60 Hz)
FS_ADC = F_LINE * N_CYCLE           # 15,360 Hz at 60 Hz (12,800 Hz at 50 Hz)
OVERSAMPLE = 8
FS_SIM = FS_ADC * OVERSAMPLE        # 122,880 Hz at 60 Hz (102,400 Hz at 50 Hz)
CT_RATIO = 2000.0                   # SCT-013-000 clamp CT (100 A : 50 mA)
R_BURDEN = 33.0
GAIN_V_PER_A = R_BURDEN / CT_RATIO  # 16.5 mV per primary amp
VREF, VDD = 1.65, 3.3
ADC_BITS = 12
ADC_FS_V = 3.3
ADC_NOISE_LSB = 1.5                 # assumed ADC noise; ESP32 nonlinearity is not modeled
# (R, C_feedback, C_ground) of each unity-gain Sallen-Key stage (matches circuit/front_end_ac.cir)
STAGES = ((8.66e3, 10e-9, 8.2e-9), (4.42e3, 47e-9, 6.8e-9))
# Optional 50 Hz resistor-retuned BOM (E96 values scaling R by 60/50 = 1.2x; same PCB & capacitors)
STAGES_50HZ_RETUNED = ((10.5e3, 10e-9, 8.2e-9), (5.36e3, 47e-9, 6.8e-9))


def adc_rate_hz(f_line=F_LINE):
    """ADC sampling rate (Hz) to keep exactly N_CYCLE=256 samples per power cycle."""
    return float(f_line) * N_CYCLE


def sim_rate_hz(f_line=F_LINE):
    """Oversampled analog simulation rate (Hz) for a given line frequency."""
    return adc_rate_hz(f_line) * OVERSAMPLE


def analog_den(stages=STAGES):
    """Denominator of H(s) = 1 / prod(s^2 R^2 C1 C2 + 2 s R C2 + 1)."""
    a = np.array([1.0])
    for r, c_fb, c_gnd in stages:
        a = np.polymul(a, [r * r * c_fb * c_gnd, 2 * r * c_gnd, 1.0])
    return a


def cutoff_hz(stages=STAGES):
    w = 2 * np.pi * np.logspace(2, 4.5, 20000)
    _, h = signal.freqs([1.0], analog_den(stages), worN=w)
    return float(w[np.argmax(np.abs(h) < 1 / np.sqrt(2))] / (2 * np.pi))


def frequency_response_summary(f_line=F_LINE, stages=STAGES):
    """Exact analytical response of the analog anti-alias front end at a given grid frequency."""
    f_line = float(f_line)
    fs_adc = adc_rate_hz(f_line)
    f_nyq = fs_adc / 2.0
    den = analog_den(stages)

    def eval_f(freq_hz):
        _, h = signal.freqs([1.0], den, worN=[2 * np.pi * float(freq_hz)])
        return complex(h[0])

    h_f1 = eval_f(f_line)
    h_f3 = eval_f(3.0 * f_line)
    h_1k = eval_f(1000.0)
    h_nyq = eval_f(f_nyq)
    h_fs = eval_f(fs_adc)
    fc = cutoff_hz(stages)
    return {
        "f_line_Hz": f_line,
        "samples_per_cycle": N_CYCLE,
        "adc_sample_rate_Hz": fs_adc,
        "nyquist_Hz": f_nyq,
        "window_8cycles_ms": round(8.0 / f_line * 1000.0, 2),
        "f_minus3dB_Hz": round(fc, 2),
        "harmonics_within_3dB": int(np.floor(fc / f_line)),
        "gain_V_per_A_at_f1": round(abs(h_f1) * GAIN_V_PER_A, 7),
        "gain_dB_at_3rd_vs_f1": round(float(20 * np.log10(abs(h_f3) / abs(h_f1))), 4),
        "gain_dB_at_1kHz_vs_f1": round(float(20 * np.log10(abs(h_1k) / abs(h_f1))), 4),
        "phase_lag_at_f1_deg": round(float(-np.degrees(np.angle(h_f1))), 3),
        "attenuation_at_nyquist_dB": round(float(-20 * np.log10(abs(h_nyq))), 2),
        "attenuation_at_fs_dB": round(float(-20 * np.log10(abs(h_fs))), 2),
    }


_SOS_CACHE = {}


def _sos_for_rate(fs_sim, stages=STAGES):
    key = (float(fs_sim), stages)
    if key not in _SOS_CACHE:
        a = analog_den(stages)
        z, p, k = signal.bilinear_zpk([], np.roots(a), 1.0 / a[0], float(fs_sim))
        _SOS_CACHE[key] = signal.zpk2sos(z, p, k)
    return _SOS_CACHE[key]


def _sos():
    return _sos_for_rate(FS_SIM, STAGES)


def sense(i_primary, rng, f_line=F_LINE, stages=STAGES):
    """Primary current (A) at sim_rate_hz(f_line) -> 12-bit ADC codes at adc_rate_hz(f_line)."""
    sos = _sos_for_rate(sim_rate_hz(f_line), stages)
    v = signal.sosfilt(sos, i_primary * GAIN_V_PER_A)[::OVERSAMPLE] + VREF
    v = np.clip(v, -0.3, VDD + 0.3)                      # Schottky clamps
    lsb = ADC_FS_V / (2 ** ADC_BITS - 1)
    code = np.round(v / lsb + rng.normal(0.0, ADC_NOISE_LSB, v.shape))
    return np.clip(code, 0, 2 ** ADC_BITS - 1).astype(np.int16)


def codes_to_amps(code):
    lsb = ADC_FS_V / (2 ** ADC_BITS - 1)
    return (code.astype(np.float64) * lsb - VREF) / GAIN_V_PER_A


In [ ]:
%%writefile hif/waveforms.py
"""Residual-current waveforms: normal feeder operation, disturbances, and Emanuel-model HIF arcs.

All parameter ranges are assumptions chosen to cover typical published behaviour; they are not field
measurements. Results trained on these waveforms must be described as simulation results.
"""
import numpy as np

from .frontend import F_LINE, FS_SIM, N_CYCLE, OVERSAMPLE, codes_to_amps, sense, sim_rate_hz

W = 2 * np.pi * F_LINE
TWO_PI = 2 * np.pi
V_PHASE_PK = 7200.0 * np.sqrt(2)    # 12.47 kV feeder, phase-to-ground peak
WARMUP = 4                          # cycles dropped while the filter settles
N_CYCLES = 60                       # cycles kept per event (1 s at 60 Hz, 1.2 s at 50 Hz)

# Emanuel arc model: conducts above +Vp through Rp and below -Vn through Rn.
SURFACES = {
    "wet_grass": {"Vp": (0.5e3, 2.0e3), "R": (300.0, 700.0)},
    "wet_sand": {"Vp": (1.0e3, 3.0e3), "R": (600.0, 1500.0)},
    "concrete": {"Vp": (1.5e3, 4.5e3), "R": (800.0, 2000.0)},
    "dry_grass": {"Vp": (2.0e3, 5.0e3), "R": (900.0, 2500.0)},
    "dry_asphalt": {"Vp": (3.0e3, 7.0e3), "R": (2000.0, 6000.0)},
}
DISTURBANCES = ("none", "load_step", "cap_switch", "inrush", "motor_start", "arc_welder",
                "nonlinear_fluct")


def _base(t, rng, w=W):
    a1 = np.exp(rng.uniform(np.log(1.0), np.log(20.0))) * np.sqrt(2)   # 1-20 A rms unbalance
    ph = rng.uniform(0, TWO_PI)
    env = 1 + rng.uniform(0, 0.04) * np.sin(TWO_PI * rng.uniform(0.1, 2.0) * t + rng.uniform(0, TWO_PI))
    i = a1 * env * np.sin(w * t + ph)
    for h, hi in ((3, 0.6), (5, 0.2), (7, 0.1), (9, 0.08)):          # triplen-heavy neutral current
        i += a1 * rng.uniform(0, hi) * np.sin(h * w * t + rng.uniform(0, TWO_PI))
    i += rng.normal(0, rng.uniform(0.02, 0.2), t.shape)
    if rng.random() < 0.3:                                              # inverter switching noise
        i += rng.uniform(0.2, 2.0) * np.sin(TWO_PI * rng.uniform(8e3, 20e3) * t + rng.uniform(0, TWO_PI))
    return i, a1, ph


def _arc(t, t0, rng, s, w=W, v_phase_pk=V_PHASE_PK):
    vp, rp = rng.uniform(*s["Vp"]), rng.uniform(*s["R"])
    vn, rn = vp * rng.uniform(1.05, 1.4), rp * rng.uniform(0.85, 1.15)
    if rng.random() < 0.5:
        vp, vn, rp, rn = vn, vp, rn, rp
    sig, p_off = rng.uniform(0.03, 0.12), rng.uniform(0.0, 0.3)
    phv = rng.uniform(0, TWO_PI)
    v = v_phase_pk * np.sin(w * t + phv)
    half = np.floor((w * t + phv) / np.pi).astype(np.int64)
    half -= half.min()
    jit = 1 + sig * rng.standard_normal((4, half.max() + 1))            # new arc each half cycle
    VP, VN, RP, RN = vp * jit[0][half], vn * jit[1][half], rp * jit[2][half], rn * jit[3][half]
    i = np.where(v > VP, (v - VP) / RP, 0.0) + np.where(v < -VN, (v + VN) / RN, 0.0)
    cyc = half // 2
    i *= (rng.random(cyc.max() + 1) >= p_off)[cyc]                      # arc fails to restrike
    dt = t - t0
    a0, tb = rng.uniform(0.1, 0.5), rng.uniform(0.02, 0.3)
    env = np.where(dt >= 0, 1 - (1 - a0) * np.exp(-np.clip(dt, 0, None) / tb), 0.0)
    return i * env


def _bursts(rng, n):
    on, state, k = np.zeros(n, bool), rng.random() < 0.5, 0
    while k < n:
        L = int(rng.integers(6, 31))
        on[k:k + L] = state
        state, k = not state, k + L
    return on


def _disturb(t, t1, rng, kind, a1, ph, w=W):
    if kind == "none":
        return 0.0
    dt = t - t1
    on = dt >= 0
    dtp = np.clip(dt, 0, None)
    cyc = np.floor(w * t / TWO_PI).astype(np.int64)
    nc = cyc.max() + 1
    if kind == "load_step":
        return on * rng.uniform(-0.5, 1.0) * a1 * np.sin(w * t + ph + rng.uniform(-0.3, 0.3))
    if kind == "cap_switch":
        A, f, tau = rng.uniform(5, 40), rng.uniform(300, 1200), rng.uniform(2e-3, 10e-3)
        return on * A * np.exp(-dtp / tau) * np.sin(TWO_PI * f * dtp + rng.uniform(0, TWO_PI))
    if kind == "inrush":
        ipk, tau, th = rng.uniform(10, 40), rng.uniform(0.1, 1.0), np.deg2rad(rng.uniform(60, 150))
        pulse = np.clip(np.sin(w * t + rng.uniform(0, TWO_PI)) - np.cos(th), 0, None) / (1 - np.cos(th))
        return on * rng.choice([-1.0, 1.0]) * ipk * np.exp(-dtp / tau) * pulse
    if kind == "motor_start":
        k, tau, lag = rng.uniform(2, 6), rng.uniform(0.1, 0.5), np.deg2rad(rng.uniform(40, 80))
        return on * (k - 1) * a1 * np.exp(-dtp / tau) * np.sin(w * t + ph - lag)
    if kind == "arc_welder":                                            # hard negative: random but symmetric
        aw = rng.uniform(3, 20) * np.sqrt(2)
        amp = np.clip(1 + 0.25 * rng.standard_normal(nc), 0.2, None)
        x = np.tanh(4 * np.sin(w * t + rng.uniform(0, TWO_PI)))
        return on * aw * amp[cyc] * _bursts(rng, nc)[cyc] * x
    if kind == "nonlinear_fluct":                                       # many electronic loads switching
        out = 0.0
        for h, hi in ((3, 0.5), (5, 0.2), (7, 0.1)):
            fac = 1 + rng.uniform(0.1, 0.3) * rng.standard_normal(nc)
            out = out + a1 * rng.uniform(0.1, hi) * fac[cyc] * np.sin(h * w * t + rng.uniform(0, TWO_PI))
        return on * out
    raise ValueError(kind)


def make_event(rng, is_hif, surface=None, disturbance=None, f_line=F_LINE):
    """One 60-cycle record. Returns (amps at the ADC, meta). meta['start_cycle'] is the event cycle."""
    fs_sim = FS_SIM if f_line == F_LINE else sim_rate_hz(f_line)
    w = W if f_line == F_LINE else 2 * np.pi * float(f_line)
    n = (WARMUP + N_CYCLES) * N_CYCLE * OVERSAMPLE
    t = np.arange(n) / fs_sim
    start = int(rng.integers(12, 41))
    t0 = (WARMUP + start) / float(f_line)
    i, a1, ph = _base(t, rng, w=w)
    meta = {"is_hif": bool(is_hif), "start_cycle": start, "base_rms": float(a1 / np.sqrt(2))}
    if f_line != F_LINE:
        meta["f_line"] = float(f_line)
    if is_hif:
        surface = surface or str(rng.choice(list(SURFACES)))
        arc = _arc(t, t0, rng, SURFACES[surface], w=w)
        i = i + arc
        if disturbance is None:
            disturbance = str(rng.choice(DISTURBANCES)) if rng.random() < 0.25 else "none"
        t1 = (WARMUP + rng.integers(0, N_CYCLES)) / float(f_line)
        tail = arc[-20 * N_CYCLE * OVERSAMPLE:]
        meta["arc_rms"] = float(np.sqrt(np.mean(tail ** 2)))
    else:
        disturbance = disturbance or str(rng.choice(DISTURBANCES))
        t1 = t0
        meta["arc_rms"] = 0.0
    i = i + _disturb(t, t1, rng, disturbance, a1, ph, w=w)
    meta["surface"] = surface or "none"
    meta["disturbance"] = disturbance
    amps = codes_to_amps(sense(i, rng, f_line=f_line))[WARMUP * N_CYCLE:]
    return amps, meta


In [ ]:
%%writefile hif/features.py
"""16 window features computed from the last K cycles of ADC samples (cheap enough for an MCU)."""
import numpy as np

from .frontend import N_CYCLE

K = 8
FEATURES = ["log_I1", "log_I3", "h2_ratio", "h3_ratio", "h5_ratio", "even_ratio", "dc_ratio",
            "peak_asym", "I3_cv", "I1_cv", "ph3_jitter", "cycle_diff", "even_std", "h3_trend",
            "I1_trend", "interharm"]
EPS = 1e-6


def window_features(amps):
    """amps: 1-D samples, whole cycles. Returns (X [n_windows, 16], end_cycle [n_windows])."""
    c = amps.reshape(-1, N_CYCLE)
    X = np.fft.rfft(c, axis=1) / (N_CYCLE / 2)
    X[:, 0] /= 2
    mag = np.abs(X)
    I1 = mag[:, 1] + EPS
    I3 = mag[:, 3] + EPS
    even = mag[:, 2] + mag[:, 4] + mag[:, 6]
    asym = (c.max(1) + c.min(1)) / (c.max(1) - c.min(1) + EPS)
    ph31 = np.exp(1j * (np.angle(X[:, 3]) - 3 * np.angle(X[:, 1])))
    energy = (c ** 2).sum(1) + EPS
    cdiff = np.zeros(len(c))
    cdiff[1:] = ((c[1:] - c[:-1]) ** 2).sum(1) / energy[1:]
    rows, ends = [], []
    h = K // 2
    for j in range(K - 1, len(c)):
        s = slice(j - K + 1, j + 1)
        i1, i3 = I1[s], I3[s]
        R = min(max(abs(ph31[s].mean()), EPS), 1.0)
        P = np.abs(np.fft.rfft(c[j - 3:j + 1].ravel())) ** 2
        rows.append([
            np.log10(i1.mean()), np.log10(i3.mean()),
            (mag[s, 2] / i1).mean(), (i3 / i1).mean(), (mag[s, 5] / i1).mean(),
            (even[s] / i1).mean(), (mag[s, 0] / i1).mean(), asym[s].mean(),
            i3.std() / i3.mean(), i1.std() / i1.mean(), np.sqrt(-2 * np.log(R)),
            cdiff[s].mean(), (even[s] / i1).std(),
            (i3[h:] / i1[h:]).mean() - (i3[:h] / i1[:h]).mean(),
            np.log10(i1[h:].mean() / i1[:h].mean()),
            1 - P[::4].sum() / (P.sum() + EPS),
        ])
        ends.append(j)
    return np.asarray(rows, np.float32), np.asarray(ends, np.int16)


def window_labels(ends, meta):
    """1 = at least K/2 arcing cycles in the window, 0 = no arc yet / normal, -1 = ambiguous (skipped)."""
    if not meta["is_hif"]:
        return np.zeros(len(ends), np.int8)
    s = meta["start_cycle"]
    y = np.full(len(ends), -1, np.int8)
    y[ends < s] = 0
    y[ends - s + 1 >= K // 2] = 1
    return y


In [ ]:
%%writefile hif/provenance.py
"""Cryptographic SHA-256 provenance manifest recorder and verifier for hif-sensor."""
import argparse
import datetime as dt
import hashlib
import json
import subprocess
import sys
from pathlib import Path

ROOT = Path(__file__).resolve().parents[1]
MANIFEST_PATH = ROOT / "results" / "provenance" / "manifest.json"

TRACKED_FILES = [
    ("circuit/ltspice_results.json", "LTspice 26.1.1 AC/transient front-end verification"),
    ("results/metrics.json", "Phase 0 simulated training/test/LOSO metrics (Kaggle christoopherchau/ch-y-n)"),
    ("results/model_int8.npz", "Phase 0 frozen 504 B int8 Tiny MLP weights + baseline pickups"),
    ("results/test_vectors.json", "Phase 0 Python reference test vectors for C firmware parity"),
    ("results/real_vehif.json", "Phase 1 zero-shot VeHIF 22 kV evaluation (Kaggle christoopherchau/ch-y-n)"),
    ("results/real_vehif_tests.csv", "Phase 1 per-recording VeHIF 22 kV results (250 recordings)"),
    ("firmware/hif_model.h", "Phase 0 auto-generated C int8 model header"),
    ("results/c_firmware_verify.json", "Phase 1.5 GCC -O2 -Wall -Werror C firmware parity & latency check"),
    ("results/phase15_vehif.json", "Phase 1.5 VeHIF adaptation, feature ablation & 50/60 Hz study"),
    ("results/model_phase15_int8.npz", "Phase 1.5 few-shot adapted 504 B int8 Tiny MLP"),
    ("firmware/hif_model_phase15.h", "Phase 1.5 adapted C int8 model header"),
]


def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(65536), b""):
            h.update(chunk)
    return h.hexdigest()


def git_head() -> str | None:
    try:
        out = subprocess.check_output(
            ["git", "rev-parse", "--short", "HEAD"], cwd=ROOT, stderr=subprocess.DEVNULL
        )
        return out.decode().strip() or None
    except Exception:
        return None


def record_manifest(root: Path = ROOT, out_path: Path | None = None) -> dict:
    out_path = out_path or (root / "results" / "provenance" / "manifest.json")
    entries = []
    seen = set()
    for rel, desc in TRACKED_FILES:
        p = root / rel
        if p.exists():
            entries.append({
                "path": rel,
                "sha256": sha256_file(p),
                "bytes": p.stat().st_size,
                "description": desc,
            })
            seen.add(rel)
    fig_dir = root / "results" / "figures"
    if fig_dir.exists():
        for p in sorted(fig_dir.glob("*.png")):
            rel = p.relative_to(root).as_posix()
            if rel not in seen:
                entries.append({
                    "path": rel,
                    "sha256": sha256_file(p),
                    "bytes": p.stat().st_size,
                    "description": f"Generated result figure ({p.name})",
                })
    manifest = {
        "project": "hif-sensor",
        "generated_utc": dt.datetime.now(dt.timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "git_head": git_head(),
        "file_count": len(entries),
        "files": entries,
    }
    out_path.parent.mkdir(parents=True, exist_ok=True)
    out_path.write_text(json.dumps(manifest, indent=2) + "\n", encoding="utf-8")
    return manifest


def verify_manifest(root: Path = ROOT, manifest_path: Path | None = None) -> dict:
    manifest_path = manifest_path or (root / "results" / "provenance" / "manifest.json")
    if not manifest_path.exists():
        return {"ok": False, "error": f"Manifest missing: {manifest_path}", "checked": 0,
                "changed": [], "missing": [str(manifest_path)]}
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    changed, missing = [], []
    for item in manifest.get("files", []):
        p = root / item["path"]
        if not p.exists():
            missing.append(item["path"])
            continue
        actual = sha256_file(p)
        if actual != item["sha256"]:
            changed.append({"path": item["path"], "expected": item["sha256"], "actual": actual})
    return {
        "ok": not changed and not missing,
        "checked": len(manifest.get("files", [])),
        "changed": changed,
        "missing": missing,
        "generated_utc": manifest.get("generated_utc"),
    }


def main():
    ap = argparse.ArgumentParser(description="Record or verify SHA-256 provenance for hif-sensor results.")
    ap.add_argument("--record", action="store_true", help="Write results/provenance/manifest.json")
    ap.add_argument("--verify", action="store_true", help="Verify all files against manifest.json")
    args = ap.parse_args()
    if args.record:
        m = record_manifest()
        print(f"Recorded {m['file_count']} files -> {MANIFEST_PATH}")
        return 0
    res = verify_manifest()
    print(f"Verified {res['checked']} files: {len(res['changed'])} changed, {len(res['missing'])} missing")
    if not res["ok"]:
        if res.get("error"):
            print("ERROR:", res["error"])
        for c in res["changed"]:
            print("  CHANGED:", c["path"])
        for m in res["missing"]:
            print("  MISSING:", m)
        return 1
    return 0


if __name__ == "__main__":
    sys.exit(main())


In [ ]:
%%writefile train_kaggle.py
"""Phase 0 of the HIF sensor: simulate events, train a tiny MLP, quantize to int8, evaluate, export.

Run on Kaggle (CPU is enough). Writes results/metrics.json, results/figures/*.png,
firmware/hif_model.h and results/test_vectors.json.
"""
import json
import sys
import time
from pathlib import Path

import matplotlib
if "ipykernel" not in sys.modules:
    matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import roc_auc_score
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler

from hif.features import FEATURES, K, window_features, window_labels
from hif.frontend import cutoff_hz
from hif.waveforms import DISTURBANCES, SURFACES, make_event

SEED = 42
N_EVENTS = 4000            # half HIF, half normal
MAX_FALSE_ALARM = 0.01     # share of normal records allowed to alarm (picked on validation)
PERSIST = 3                # consecutive windows above threshold before an alarm (like a relay timer)
HIDDEN = (16, 8)
OUT = Path("results")
FIG = OUT / "figures"
FW = Path("firmware")
for d in (OUT, FIG, FW):
    d.mkdir(parents=True, exist_ok=True)


def show(fig, name):
    fig.tight_layout()
    fig.savefig(FIG / name, dpi=120)
    if "ipykernel" in sys.modules:
        plt.show()
    plt.close(fig)


# ---------------------------------------------------------------- 1. simulate
def simulate():
    rng = np.random.default_rng(SEED)
    events = []
    t0 = time.time()
    for k in range(N_EVENTS):
        amps, meta = make_event(rng, is_hif=(k % 2 == 0))
        X, ends = window_features(amps)
        events.append({"X": X, "ends": ends, "y": window_labels(ends, meta), **meta})
        if (k + 1) % 500 == 0:
            print(f"  simulated {k + 1}/{N_EVENTS} events ({time.time() - t0:.0f} s)")
    return events


# ---------------------------------------------------------------- 2. model helpers
def stack(evs):
    X = np.concatenate([e["X"] for e in evs])
    y = np.concatenate([e["y"] for e in evs])
    keep = y >= 0
    return X[keep], y[keep]


def fit(evs):
    X, y = stack(evs)
    sc = StandardScaler().fit(X)
    clf = MLPClassifier(hidden_layer_sizes=HIDDEN, alpha=1e-3, max_iter=400, early_stopping=True,
                        n_iter_no_change=15, random_state=SEED).fit(sc.transform(X), y)
    return sc, clf


def fold_and_quantize(sc, clf):
    """Fold the scaler into layer 1, then int8 weights (per-layer scale) + float32 biases."""
    Ws = [w.copy() for w in clf.coefs_]
    bs = [b.copy() for b in clf.intercepts_]
    bs[0] = bs[0] - (sc.mean_ / sc.scale_) @ Ws[0]
    Ws[0] = Ws[0] / sc.scale_[:, None]
    layers = []
    for W, b in zip(Ws, bs):
        s = float(np.abs(W).max() / 127.0)
        layers.append({"Wq": np.round(W / s).astype(np.int8), "scale": s, "b": b.astype(np.float32)})
    return layers


def predict_q(layers, X):
    h = X.astype(np.float32)
    for n, L in enumerate(layers):
        h = h @ (L["Wq"].astype(np.float32) * L["scale"]) + L["b"]
        h = np.maximum(h, 0) if n < len(layers) - 1 else 1 / (1 + np.exp(-h))
    return h.ravel()


def alarms(p, thr):
    run, out = 0, []
    for k, a in enumerate(p > thr):
        run = run + 1 if a else 0
        if run == PERSIST:
            out.append(k)
    return out


def event_result(e, score, thr):
    al = [int(e["ends"][k]) for k in alarms(score, thr)]
    s = e["start_cycle"]
    if e["is_hif"]:
        pre = any(a < s for a in al)
        post = [a for a in al if a >= s]
        return {"false_alarm": pre, "detected": bool(post), "delay": (post[0] - s) if post else None}
    return {"false_alarm": bool(al), "detected": False, "delay": None}


def pick_threshold(evs, scores, grid):
    """Lowest threshold whose false-alarm rate is within budget (FA falls as thr rises: bisection)."""
    def fa(thr):
        return np.mean([event_result(e, s, thr)["false_alarm"] for e, s in zip(evs, scores)])
    lo, hi = 0, len(grid) - 1
    if fa(grid[hi]) > MAX_FALSE_ALARM:
        return float(grid[hi])
    while lo < hi:
        mid = (lo + hi) // 2
        if fa(grid[mid]) <= MAX_FALSE_ALARM:
            hi = mid
        else:
            lo = mid + 1
    return float(grid[lo])


def summarize(evs, scores, thr):
    res = [event_result(e, s, thr) for e, s in zip(evs, scores)]
    hif = [(e, r) for e, r in zip(evs, res) if e["is_hif"]]
    nor = [(e, r) for e, r in zip(evs, res) if not e["is_hif"]]
    delays = [r["delay"] for _, r in hif if r["delay"] is not None]
    out = {
        "hif_records": len(hif), "normal_records": len(nor),
        "detection_rate": float(np.mean([r["detected"] for _, r in hif])),
        "false_alarm_rate_normal": float(np.mean([r["false_alarm"] for _, r in nor])),
        "pre_fault_false_alarm_rate_hif": float(np.mean([r["false_alarm"] for _, r in hif])),
        "delay_cycles_median": float(np.median(delays)) if delays else None,
        "delay_cycles_p90": float(np.percentile(delays, 90)) if delays else None,
        "by_surface": {s: float(np.mean([r["detected"] for e, r in hif if e["surface"] == s]))
                       for s in SURFACES},
        "false_alarm_by_disturbance": {d: float(np.mean([r["false_alarm"] for e, r in nor
                                                        if e["disturbance"] == d]))
                                       for d in DISTURBANCES},
    }
    bins = [(0, 1), (1, 2), (2, 5), (5, 10), (10, 1e9)]
    out["by_arc_current_A_rms"] = {}
    for lo, hi in bins:
        sel = [r["detected"] for e, r in hif if lo <= e["arc_rms"] < hi]
        key = f"{lo}-{hi}" if hi < 1e9 else f">{lo}"
        out["by_arc_current_A_rms"][key] = {"n": len(sel), "detected": float(np.mean(sel)) if sel else None}
    return out


def split(evs, rng):
    idx = rng.permutation(len(evs))
    n1, n2 = int(0.6 * len(evs)), int(0.8 * len(evs))
    return [evs[i] for i in idx[:n1]], [evs[i] for i in idx[n1:n2]], [evs[i] for i in idx[n2:]]


def train_eval(tr, va, te):
    sc, clf = fit(tr)
    layers = fold_and_quantize(sc, clf)
    grid = np.linspace(0.30, 0.999, 300)
    thr = pick_threshold(va, [predict_q(layers, e["X"]) for e in va], grid)
    return layers, thr, summarize(te, [predict_q(layers, e["X"]) for e in te], thr)


# ---------------------------------------------------------------- 3. main
def main():
    t_start = time.time()
    print(f"front-end model -3 dB = {cutoff_hz():.1f} Hz (LTspice: 2045.1 Hz)")
    events = simulate()
    rng = np.random.default_rng(SEED)
    tr, va, te = split(events, rng)
    print(f"events train/val/test = {len(tr)}/{len(va)}/{len(te)}")

    # main model
    sc, clf = fit(tr)
    layers = fold_and_quantize(sc, clf)
    grid = np.linspace(0.30, 0.999, 300)
    thr = pick_threshold(va, [predict_q(layers, e["X"]) for e in va], grid)
    te_q = [predict_q(layers, e["X"]) for e in te]
    te_f = [clf.predict_proba(sc.transform(e["X"]))[:, 1] for e in te]
    ml = summarize(te, te_q, thr)
    Xt, yt = stack(te)
    pq = predict_q(layers, Xt)
    pf = clf.predict_proba(sc.transform(Xt))[:, 1]
    n_params = sum(L["Wq"].size + L["b"].size for L in layers)
    model_bytes = sum(L["Wq"].size + 4 * L["b"].size + 4 for L in layers)

    # classical baselines, same persistence logic and same false-alarm budget
    base = {}
    for name, col in (("residual_overcurrent_51N", "log_I1"), ("third_harmonic_level", "log_I3")):
        j = FEATURES.index(col)
        g = np.linspace(-1.0, 2.5, 400)
        bthr = pick_threshold(va, [e["X"][:, j] for e in va], g)
        base[name] = summarize(te, [e["X"][:, j] for e in te], bthr)
        base[name]["pickup_A_peak"] = float(10 ** bthr)

    # generalization: leave one surface out, and an unseen hard-negative load
    loso = {}
    for s in SURFACES:
        keep = lambda e: not (e["is_hif"] and e["surface"] == s)  # noqa: E731
        _, _, r = train_eval([e for e in tr if keep(e)], [e for e in va if keep(e)], te)
        loso[s] = {"detection_unseen_surface": r["by_surface"][s],
                   "false_alarm_rate_normal": r["false_alarm_rate_normal"]}
        print(f"  leave-out {s}: detection {r['by_surface'][s]:.3f}")
    keep = lambda e: e["disturbance"] != "arc_welder" or e["is_hif"]  # noqa: E731
    _, _, r = train_eval([e for e in tr if keep(e)], [e for e in va if keep(e)], te)
    unseen_welder = {"false_alarm_arc_welder_unseen": r["false_alarm_by_disturbance"]["arc_welder"],
                     "false_alarm_arc_welder_seen": ml["false_alarm_by_disturbance"]["arc_welder"]}

    metrics = {
        "note": "All waveforms are simulated (Emanuel arc model + assumed load/disturbance ranges). "
                "Not field data.",
        "setup": {"events": N_EVENTS, "cycles_per_event": 60, "window_cycles": K,
                  "features": FEATURES, "hidden": list(HIDDEN), "persist_windows": PERSIST,
                  "false_alarm_budget_val": MAX_FALSE_ALARM, "seed": SEED,
                  "windows_total": int(sum(len(e["y"]) for e in events))},
        "frontend_cutoff_hz_python": cutoff_hz(),
        "model": {"params": int(n_params), "bytes_int8_weights_float_bias": int(model_bytes),
                  "threshold": thr, "window_auc_int8": float(roc_auc_score(yt, pq)),
                  "window_auc_float": float(roc_auc_score(yt, pf)),
                  "window_decision_agreement_int8_vs_float": float(np.mean((pq > thr) == (pf > thr))),
                  "event_detection_float": summarize(te, te_f, thr)["detection_rate"]},
        "test_ml_int8": ml,
        "test_baselines": base,
        "leave_one_surface_out": loso,
        "unseen_arc_welder": unseen_welder,
        "runtime_s": round(time.time() - t_start, 1),
    }
    (OUT / "metrics.json").write_text(json.dumps(metrics, indent=2))
    export_firmware(layers, thr, te, te_q)
    save_model(layers, thr, base)
    figures(ml, base, loso, te)
    print_summary(metrics)


def save_model(layers, thr, base):
    """Python-loadable copy of the int8 model and the baseline pickups (read by real_data_test.py)."""
    d = {"thr": thr}
    for n, L in enumerate(layers):
        d[f"W{n}"], d[f"s{n}"], d[f"b{n}"] = L["Wq"], L["scale"], L["b"]
    for name, br in base.items():
        d["pickup_" + name] = br["pickup_A_peak"]
    np.savez(OUT / "model_int8.npz", **d)


def export_firmware(layers, thr, te, te_q):
    def carr(a, fmt):
        return "{" + ", ".join(fmt(v) for v in np.ravel(a)) + "}"
    def carr2d(a2d, fmt):
        return "{" + ", ".join("{" + ", ".join(fmt(v) for v in row) + "}" for row in a2d) + "}"
    lines = ["// Auto-generated by train_kaggle.py. Tiny MLP for HIF detection (int8 weights).",
             "#pragma once", "#include <stdint.h>",
             f"#define HIF_N_IN {layers[0]['Wq'].shape[0]}", f"#define HIF_PERSIST {PERSIST}",
             f"static const float HIF_THRESHOLD = {thr:.6f}f;"]
    dims = []
    for n, L in enumerate(layers, 1):
        Wt = L["Wq"].T                                    # [out][in]
        dims.append(Wt.shape)
        lines += [f"#define HIF_L{n}_OUT {Wt.shape[0]}", f"#define HIF_L{n}_IN {Wt.shape[1]}",
                  f"static const int8_t HIF_W{n}[{Wt.shape[0]}][{Wt.shape[1]}] = "
                  + carr2d(Wt, lambda v: str(int(v))) + ";",
                  f"static const float HIF_S{n} = {L['scale']:.8e}f;",
                  f"static const float HIF_B{n}[{Wt.shape[0]}] = " + carr(L["b"], lambda v: f"{v:.8e}f") + ";"]
    (FW / "hif_model.h").write_text("\n".join(lines) + "\n")
    vec = []
    for e, p in zip(te[:6], te_q[:6]):
        k = len(p) - 1
        vec.append({"features": e["X"][k].tolist(), "prob": float(p[k]), "is_hif": e["is_hif"]})
    (OUT / "test_vectors.json").write_text(json.dumps(vec, indent=2))


def figures(ml, base, loso, te):
    names = list(SURFACES)
    fig, ax = plt.subplots(figsize=(8, 3.5))
    w = 0.27
    x = np.arange(len(names))
    ax.bar(x - w, [ml["by_surface"][s] for s in names], w, label="tiny MLP (int8)")
    for k, (bn, br) in enumerate(base.items()):
        ax.bar(x + k * w, [br["by_surface"][s] for s in names], w, label=bn)
    ax.set_xticks(x, names)
    ax.set_ylabel("HIF records detected")
    ax.set_ylim(0, 1.05)
    ax.legend(fontsize=8)
    ax.set_title(f"Detection by surface at <= {MAX_FALSE_ALARM:.0%} false alarms (simulated)")
    show(fig, "detection_by_surface.png")

    fig, ax = plt.subplots(figsize=(8, 3.2))
    d = list(DISTURBANCES)
    ax.bar(d, [ml["false_alarm_by_disturbance"][k] for k in d])
    ax.set_ylabel("normal records with an alarm")
    ax.set_title("False alarms by normal event type (test set)")
    plt.setp(ax.get_xticklabels(), rotation=20)
    show(fig, "false_alarm_by_disturbance.png")

    fig, ax = plt.subplots(figsize=(6, 3.2))
    b = ml["by_arc_current_A_rms"]
    ax.bar(list(b), [v["detected"] or 0 for v in b.values()])
    for i, v in enumerate(b.values()):
        ax.text(i, 0.02, f"n={v['n']}", ha="center", fontsize=8, color="white")
    ax.set_xlabel("HIF arc current (A rms)")
    ax.set_ylabel("detected")
    ax.set_title("Detection vs fault current size")
    show(fig, "detection_vs_current.png")

    fig, ax = plt.subplots(figsize=(6, 3.2))
    ax.bar(names, [loso[s]["detection_unseen_surface"] for s in names], color="tab:orange")
    ax.set_ylim(0, 1.05)
    ax.set_title("Detection when the surface was never seen in training")
    plt.setp(ax.get_xticklabels(), rotation=15)
    show(fig, "leave_one_surface_out.png")


def print_summary(m):
    ml = m["test_ml_int8"]
    print("\n================ RESULTS (simulated test set) ================")
    print(f"tiny MLP int8: {m['model']['params']} params, {m['model']['bytes_int8_weights_float_bias']} bytes")
    print(f"HIF detected {ml['detection_rate']:.1%} | normal false alarms {ml['false_alarm_rate_normal']:.1%}"
          f" | median delay {ml['delay_cycles_median']} cycles (p90 {ml['delay_cycles_p90']})")
    for bn, br in m["test_baselines"].items():
        print(f"{bn}: detected {br['detection_rate']:.1%} | false alarms {br['false_alarm_rate_normal']:.1%}"
              f" | pickup {br['pickup_A_peak']:.1f} A peak")
    print("by surface:", {k: round(v, 3) for k, v in ml["by_surface"].items()})
    print("false alarm by load:", {k: round(v, 3) for k, v in ml["false_alarm_by_disturbance"].items()})
    print("leave-one-surface-out:", {k: round(v["detection_unseen_surface"], 3)
                                     for k, v in m["leave_one_surface_out"].items()})
    print("unseen arc welder:", m["unseen_arc_welder"])
    print(f"runtime {m['runtime_s']} s -> results/metrics.json, firmware/hif_model.h")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile real_data_test.py
"""Phase 1 of the HIF sensor: test the simulation-trained detector on REAL fault recordings.

Data set: VeHIF, 983 vegetation high-impedance fault tests and 53 background recordings made on a real
22 kV, 50 Hz feeder (funded by the Victorian Government, Australia). HDF5 format: D. Gomes and C. Ozansoy,
IEEE Trans. Power Delivery 37(6), 2022 (arXiv:2112.03651). Kaggle: dougpsg/highimpedance-vegetation-fault-data-set.

What is real and what is simulated here:
- REAL: the fault current waveform (LF channel) and the background recordings.
- SIMULATED: the sensor (front end + 12-bit ADC model) and the residual load current of the feeder, which is
  added on top of the real current. Each recording runs twice with the same load: without the recording
  (control) and with it. "Detected" = an alarm with the real fault and no alarm in the control.
- The detector is the Phase 0 model, trained on simulated arcs only. Nothing is re-trained here.
The 50 Hz recordings are resampled to 256 samples per cycle so harmonic orders match the 60 Hz design.
Vegetation faults are not downed conductors, so this measures how far the simulation-trained model transfers.
"""
import csv
import glob
import json
import os
import sys
import time
from fractions import Fraction
from pathlib import Path

import matplotlib
if "ipykernel" not in sys.modules:
    matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
from scipy import signal

from hif.features import FEATURES, K, window_features, window_labels
from hif.frontend import FS_SIM, N_CYCLE, OVERSAMPLE, codes_to_amps, sense
from hif.waveforms import SURFACES, WARMUP, _arc, _base, make_event
from train_kaggle import alarms, predict_q

H5_NAME = "hif_vegetation_dataset.h5"
N_TESTS = 250            # fault recordings to read; None = every valid one (slower)
MAX_SECONDS = 120        # only the last part of a long recording is analysed
MAX_MINUTES = 45         # stop reading new recordings after this long
GRID_HZ = 50.0
SEED = 7
SPC = N_CYCLE * OVERSAMPLE                                      # 2048 samples per cycle go into the sensor model
BINS = [(0, 1), (1, 2), (2, 5), (5, 10), (10, 1e9)]             # fault current, A rms (same bins as Phase 0)
RATIO_BINS = [(0, 0.05), (0.05, 0.2), (0.2, 0.5), (0.5, 1e9)]   # fault current / load current
J1, J3 = FEATURES.index("log_I1"), FEATURES.index("log_I3")
OUT = Path("results")
FIG = OUT / "figures"


# ---------------------------------------------------------------- reading the file
def find_h5():
    p = os.environ.get("VEHIF_PATH")
    if p and Path(p).exists():
        return p
    hits = glob.glob("/kaggle/input/**/" + H5_NAME, recursive=True)
    hits = hits or glob.glob("/kaggle/input/**/*.h5", recursive=True)
    return hits[0] if hits else None


def load_model(path=None):
    z = np.load(path or OUT / "model_int8.npz")
    layers, n = [], 0
    while f"W{n}" in z.files:
        layers.append({"Wq": z[f"W{n}"], "scale": float(z[f"s{n}"]), "b": z[f"b{n}"]})
        n += 1
    pick = {k[len("pickup_"):]: float(z[k]) for k in z.files if k.startswith("pickup_")}
    return layers, float(z["thr"]), pick


def _txt(v):
    if isinstance(v, np.ndarray):
        v = v.ravel()[0] if v.size else ""
    if isinstance(v, bytes):
        v = v.decode(errors="ignore")
    return str(v).strip().strip("\x00")


def attr(g, *names, num=False):
    for n in names:
        if n in g.attrs:
            v = g.attrs[n]
            if not num:
                return _txt(v)
            try:
                a = np.asarray(v, np.float64).ravel()
                return float(a[0]) if a.size and np.isfinite(a[0]) and a[0] > 0 else None
            except (TypeError, ValueError):
                return None
    return None


def _axis_slice(ds, lo, hi):
    ax = int(np.argmax(ds.shape))
    sl = [slice(None)] * len(ds.shape)
    sl[ax] = slice(lo, hi)
    return np.asarray(ds[tuple(sl)], np.float64).ravel()


def read_head(ds, n):
    return _axis_slice(ds, 0, min(n, max(ds.shape)))


def read_tail(ds, n):
    """Last n samples of a 1-D data set stored as (N,), (N, 1) or (1, N). Returns (samples, N)."""
    N = max(ds.shape)
    return _axis_slice(ds, max(0, N - n), N), N


# ---------------------------------------------------------------- signal helpers
def samples_per_cycle(v, guess=2000.0):
    """Length of one power cycle in samples, measured on the voltage (fallback: 100 kSa/s at 50 Hz)."""
    x = np.asarray(v, np.float64)
    if x.size < 20000 or not np.isfinite(x).all() or x.std() == 0:
        return guess
    X = np.abs(np.fft.rfft((x - x.mean()) * np.hanning(x.size)))
    lo, hi = int(x.size / (guess * 1.5)), int(x.size / (guess / 1.5))
    k = lo + int(np.argmax(X[lo:hi]))
    a, b, c = X[k - 1], X[k], X[k + 1]
    spc = x.size / (k + 0.5 * (a - c) / (a - 2 * b + c - 1e-30))
    return float(spc) if abs(spc / guess - 1) < 0.2 else guess


def to_sensor_rate(x, spc):
    """Resample so that one power cycle = 2048 samples (the rate the sensor model expects)."""
    fr = Fraction(SPC / spc).limit_denominator(1000)
    return signal.resample_poly(x, fr.numerator, fr.denominator)


def cycle_ms(x):
    return (x.reshape(-1, SPC) ** 2).mean(1)


def rolling_rms(ms, n):
    if len(ms) < n:
        return np.sqrt(ms.mean(keepdims=True))
    c = np.cumsum(np.insert(ms, 0, 0.0))
    return np.sqrt((c[n:] - c[:-n]) / n)


def through_sensor(x, rng):
    """Primary current at 2048 samples/cycle -> sensor model -> 16 features per window."""
    amps = codes_to_amps(sense(x, rng))[WARMUP * N_CYCLE:]
    X, _ = window_features(amps)
    return X, np.arange(K - 1, K - 1 + len(X)) + WARMUP     # last cycle of each window, counted in the recording


def first_alarm(score, thr):
    al = alarms(score, thr)
    return al[0] if al else None


# ---------------------------------------------------------------- one recording
def run_record(g, rng, layers, thr, base_thr):
    spc = samples_per_cycle(read_head(g["voltage_lf"], 1 << 18)) if "voltage_lf" in g else 2000.0
    raw, n_total = read_tail(g["current_lf"], int(MAX_SECONDS * GRID_HZ * spc))
    raw = np.nan_to_num(raw)
    x = to_sensor_rate(raw - raw.mean(), spc)
    n = len(x) // SPC * SPC
    if n < 40 * SPC:
        raise ValueError("recording shorter than 40 cycles")
    x = x[:n]
    load, a1, _ = _base(np.arange(n) / FS_SIM, rng)
    ms = cycle_ms(x)
    r8 = rolling_rms(ms, K)
    Xf, ends = through_sensor(load + x, rng)      # load + real recording
    Xc, _ = through_sensor(load, rng)             # control: the same load alone
    out = {"seconds": float(n_total / spc / GRID_HZ), "truncated": bool(len(raw) < n_total),
           "samples_per_cycle": float(spc), "raw_peak_A": float(np.abs(raw).max()),
           "rms_all": float(np.sqrt(ms.mean())), "peak_rms20": float(rolling_rms(ms, 20).max()),
           "load_rms": float(a1 / np.sqrt(2)), "rms_at_alarm": None}
    out["ratio"] = out["peak_rms20"] / out["load_rms"]
    for name, sf, sc, th in (("ml", predict_q(layers, Xf), predict_q(layers, Xc), thr),
                             ("51N", Xf[:, J1], Xc[:, J1], base_thr.get("residual_overcurrent_51N", 9e9)),
                             ("h3", Xf[:, J3], Xc[:, J3], base_thr.get("third_harmonic_level", 9e9))):
        kf, kc = first_alarm(sf, th), first_alarm(sc, th)
        out["alarm_" + name], out["control_" + name] = kf is not None, kc is not None
        out["det_" + name] = kf is not None and kc is None
        if name == "ml" and kf is not None:
            out["rms_at_alarm"] = float(r8[ends[kf] - (K - 1)])
    out["alarm"], out["control_alarm"], out["detected"] = out["alarm_ml"], out["control_ml"], out["det_ml"]
    strong = np.flatnonzero(r8[ends - (K - 1)] >= 1.0)          # windows with at least 1 A rms of real fault current
    out["_feat"] = Xf[strong[:: max(1, len(strong) // 150)][:150]]
    c0 = int(np.argmax(rolling_rms(ms, 3)))
    out["_snip"] = x[c0 * SPC:(c0 + 3) * SPC].astype(np.float32)
    return out


# ---------------------------------------------------------------- summaries
def sim_hif_windows(n_events):
    """Features of simulated arcing windows (same generator as Phase 0), to compare with the real ones."""
    rng = np.random.default_rng(SEED)
    rows = []
    for _ in range(n_events):
        amps, meta = make_event(rng, True)
        X, ends = window_features(amps)
        rows.append(X[window_labels(ends, meta) == 1])
    return np.concatenate(rows)


def feature_shift(real, sim):
    """How far the real arcing windows sit from the simulated ones, feature by feature."""
    out = []
    for j, name in enumerate(FEATURES):
        q25, q50, q75 = np.percentile(sim[:, j], [25, 50, 75])
        med = float(np.median(real[:, j]))
        out.append({"feature": name, "real_median": med, "sim_median": float(q50),
                    "shift_in_sim_iqr": float((med - q50) / (q75 - q25 + 1e-9))})
    return sorted(out, key=lambda d: -abs(d["shift_in_sim_iqr"]))


def by_bins(rows, key, bins, flag="detected"):
    out = {}
    for lo, hi in bins:
        sel = [r[flag] for r in rows if lo <= r[key] < hi]
        out[f"{lo:g}-{hi:g}" if hi < 1e8 else f">{lo:g}"] = {"n": len(sel),
                                                           "detected": float(np.mean(sel)) if sel else None}
    return out


def save(fig, name):
    fig.tight_layout()
    fig.savefig(FIG / name, dpi=120)
    if "ipykernel" in sys.modules:
        plt.show()
    plt.close(fig)


def figures(res, rows, snips, at):
    real = res["real_faults"]["by_peak_fault_current_A_rms"]
    sim = res.get("sim_reference_by_arc_current_A_rms") or {}
    names = list(real)
    fig, ax = plt.subplots(figsize=(7, 3.4))
    x, w = np.arange(len(names)), 0.4
    ax.bar(x - w / 2, [(sim.get(k) or {}).get("detected") or 0 for k in names], w,
           label="simulated arcs (Phase 0 test set)")
    ax.bar(x + w / 2, [real[k]["detected"] or 0 for k in names], w, label="real vegetation faults (VeHIF)")
    for k, name in enumerate(names):
        ax.text(k + w / 2, 0.02, f"n={real[name]['n']}", ha="center", fontsize=7)
    ax.set_xticks(x, names)
    ax.set_xlabel("fault current (A rms)")
    ax.set_ylabel("detected")
    ax.set_ylim(0, 1.05)
    ax.legend(fontsize=8)
    ax.set_title("Same detector: simulated arcs vs real recordings")
    save(fig, "real_detection_by_current.png")

    mid = sorted(rows, key=lambda r: r["peak_rms20"])[len(rows) // 2]["id"]
    arc = _arc(np.arange(40 * SPC) / FS_SIM, 0.0, np.random.default_rng(SEED), SURFACES["dry_grass"])[-3 * SPC:]
    fig, axs = plt.subplots(1, 2, figsize=(9, 3))
    axs[0].plot(np.arange(len(snips[mid])) / SPC, snips[mid])
    axs[0].set_title(f"Real vegetation fault (test {mid})")
    axs[1].plot(np.arange(len(arc)) / SPC, arc, color="tab:orange")
    axs[1].set_title("Simulated arc (Emanuel model, dry grass)")
    for a in axs:
        a.set_xlabel("cycles")
        a.set_ylabel("fault current (A)")
    save(fig, "real_vs_sim_waveform.png")

    if at:
        fig, ax = plt.subplots(figsize=(6, 3.2))
        ax.hist(at, bins=20)
        ax.set_xlabel("real fault current when the alarm was raised (A rms over 8 cycles)")
        ax.set_ylabel("recordings")
        ax.set_title("How much real fault current the detector needed")
        save(fig, "real_current_at_first_alarm.png")


# ---------------------------------------------------------------- main
def evaluate(f, layers, thr, pick, n_tests=N_TESTS, n_sim_events=300):
    t0 = time.time()
    FIG.mkdir(parents=True, exist_ok=True)
    base_thr = {k: float(np.log10(v)) for k, v in pick.items()}
    tg = f[next(k for k in f.keys() if k.lower().startswith("test"))]
    cg = f[next(k for k in f.keys() if k.lower().startswith("cal"))]
    ids = sorted(tg.keys())
    valid = [i for i in ids if (attr(tg[i], "report_validity") or "valid").lower() != "invalid"]
    order = np.random.default_rng(SEED).permutation(len(valid))
    chosen = sorted(valid[k] for k in (order[:n_tests] if n_tests else order))
    g0 = tg[chosen[0]]
    check = {"root_groups": sorted(f.keys()), "tests_in_file": len(ids), "tests_valid": len(valid),
             "background_in_file": len(cg.keys()), "first_test": chosen[0],
             "datasets": {k: [list(map(int, g0[k].shape)), str(getattr(g0[k], "dtype", "?"))] for k in g0.keys()},
             "attributes": {k: _txt(v)[:60] for k, v in dict(g0.attrs).items()}}
    print("FILE CHECK\n" + json.dumps(check, indent=1))

    rows, feats, snips, bg, errors = [], [], {}, [], []
    for n, i in enumerate(chosen):
        if (time.time() - t0) / 60 > MAX_MINUTES:
            print(f"time limit reached after {n} fault recordings")
            break
        try:
            g = tg[i]
            r = run_record(g, np.random.default_rng([SEED, n]), layers, thr, base_thr)
            feats.append(r.pop("_feat"))
            snips[i] = r.pop("_snip")
            r.update(id=i, fault_type=attr(g, "fault_type", "test_type", "cal_type") or "?",
                     max_current_attr=attr(g, "max_current", num=True))
            rows.append(r)
        except Exception as e:      # one unreadable recording must not stop the run
            errors.append(f"test {i}: {type(e).__name__}: {e}"[:160])
        if (n + 1) % 25 == 0:
            print(f"  {n + 1}/{len(chosen)} fault recordings ({time.time() - t0:.0f} s)")
    for n, i in enumerate(sorted(cg.keys())):
        try:
            r = run_record(cg[i], np.random.default_rng([SEED, 100000 + n]), layers, thr, base_thr)
            r.pop("_feat")
            r.pop("_snip")
            r["id"] = i
            bg.append(r)
        except Exception as e:
            errors.append(f"background {i}: {type(e).__name__}: {e}"[:160])
    if not rows:
        raise RuntimeError("no fault recording could be read: " + "; ".join(errors[:3]))

    at = [r["rms_at_alarm"] for r in rows if r["detected"] and not r["truncated"]]
    unit = [r["peak_rms20"] / r["max_current_attr"] for r in rows if r["max_current_attr"]]
    types = sorted({r["fault_type"] for r in rows})
    res = {
        "note": "REAL fault currents (VeHIF, 22 kV, 50 Hz vegetation faults) through a SIMULATED sensor with a "
                "SIMULATED load current added. Detector = Phase 0 model trained on simulated arcs only. "
                "Not a field test of hardware.",
        "data": "D. Gomes and C. Ozansoy, VeHIF, IEEE Trans. Power Delivery 37(6), 2022; "
                "kaggle.com/datasets/dougpsg/highimpedance-vegetation-fault-data-set (CC0)",
        "file_check": check,
        "setup": {"fault_recordings_requested": n_tests, "seconds_analysed_max": MAX_SECONDS, "seed": SEED,
                  "threshold": thr, "resampled_to_samples_per_cycle": N_CYCLE},
        "units_check_peak_rms_over_reported_max_current_median": float(np.median(unit)) if unit else None,
        "real_faults": {
            "recordings": len(rows),
            "detection_rate": float(np.mean([r["detected"] for r in rows])),
            "alarm_in_control_rate": float(np.mean([r["control_alarm"] for r in rows])),
            "peak_fault_current_A_rms_median": float(np.median([r["peak_rms20"] for r in rows])),
            "by_peak_fault_current_A_rms": by_bins(rows, "peak_rms20", BINS),
            "by_fault_to_load_ratio": by_bins(rows, "ratio", RATIO_BINS),
            "by_fault_type": {t: {"n": sum(r["fault_type"] == t for r in rows),
                                  "detected": float(np.mean([r["detected"] for r in rows if r["fault_type"] == t]))}
                              for t in types},
            "fault_A_rms_at_first_alarm": ({"n": len(at), "p25": float(np.percentile(at, 25)),
                                            "median": float(np.median(at)), "p75": float(np.percentile(at, 75))}
                                           if at else None),
            "baseline_detection_rate": {"residual_overcurrent_51N": float(np.mean([r["det_51N"] for r in rows])),
                                        "third_harmonic_level": float(np.mean([r["det_h3"] for r in rows]))},
        },
        "background_recordings": ({"recordings": len(bg),
                                   "alarm_rate": float(np.mean([r["alarm"] for r in bg])),
                                   "alarm_in_control_rate": float(np.mean([r["control_alarm"] for r in bg])),
                                   "noise_A_rms_median": float(np.median([r["rms_all"] for r in bg]))}
                                  if bg else None),
        "errors": errors[:20],
    }
    if (OUT / "metrics.json").exists():
        m = json.loads((OUT / "metrics.json").read_text())
        res["sim_reference_by_arc_current_A_rms"] = m.get("test_ml_int8", {}).get("by_arc_current_A_rms")
        res["sim_reference_detection_rate"] = m.get("test_ml_int8", {}).get("detection_rate")
    real = np.concatenate(feats) if feats else np.zeros((0, len(FEATURES)))
    res["real_windows_with_1A_or_more"] = int(len(real))
    if len(real) >= 50 and n_sim_events:
        res["feature_shift_real_vs_sim"] = feature_shift(real, sim_hif_windows(n_sim_events))
    res["runtime_s"] = round(time.time() - t0, 1)

    def conv(o):
        return o.item() if hasattr(o, "item") else str(o)

    (OUT / "real_vehif.json").write_text(json.dumps(res, indent=2, default=conv))
    cols = [k for k in rows[0] if not k.startswith("_")]
    with open(OUT / "real_vehif_tests.csv", "w", newline="", encoding="utf-8") as fh:
        w = csv.DictWriter(fh, fieldnames=cols, extrasaction="ignore")
        w.writeheader()
        w.writerows(rows)
    try:
        figures(res, rows, snips, at)
    except Exception as e:
        print("figures skipped:", type(e).__name__, e)

    def short(d):
        return {k: (v["n"], None if v["detected"] is None else round(v["detected"], 3)) for k, v in d.items()}

    rf = res["real_faults"]
    print("\n================ PHASE 1: REAL RECORDINGS (simulated sensor + load) ================")
    print(f"fault recordings read {rf['recordings']} | detected {rf['detection_rate']:.1%}"
          f" | alarm in control {rf['alarm_in_control_rate']:.1%}"
          f" | median peak fault current {rf['peak_fault_current_A_rms_median']:.2f} A rms")
    print("units check (measured peak / reported max current, median):",
          res["units_check_peak_rms_over_reported_max_current_median"])
    print("by peak fault current A rms (n, detected):", short(rf["by_peak_fault_current_A_rms"]))
    print("by fault/load ratio (n, detected):", short(rf["by_fault_to_load_ratio"]))
    print("by fault type (n, detected):", short(rf["by_fault_type"]))
    print("fault current at first alarm:", rf["fault_A_rms_at_first_alarm"])
    print("baselines:", rf["baseline_detection_rate"])
    print("background recordings:", res["background_recordings"])
    if "feature_shift_real_vs_sim" in res:
        print("largest real-vs-simulated feature shifts:",
              [(d["feature"], round(d["shift_in_sim_iqr"], 2)) for d in res["feature_shift_real_vs_sim"][:5]])
    if errors:
        print(f"{len(errors)} recordings skipped, first:", errors[0])
    print(f"runtime {res['runtime_s']} s -> results/real_vehif.json, results/real_vehif_tests.csv")
    return res


def main():
    if not (OUT / "model_int8.npz").exists():
        print("Phase 1 skipped: run Phase 0 first (results/model_int8.npz is missing).")
        return None
    path = find_h5()
    if path is None:
        print("Phase 1 skipped: the VeHIF data set is not attached. On Kaggle: Add Input -> search "
              "'High-impedance vegetation fault data set' (Douglas Gomes) -> +, then Run All again.")
        return None
    import h5py
    layers, thr, pick = load_model()
    print("reading", path)
    with h5py.File(path, "r") as f:
        return evaluate(f, layers, thr, pick)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile phase15_vehif_adaptation.py
"""Phase 1.5 of the HIF sensor:
1. Real GCC C firmware bit-parity & latency verification (Phase 0 & Phase 1.5 headers).
2. Explicit 50 Hz (Vietnam / Australia 22 kV) vs 60 Hz (US 12.47 kV) front-end & native-rate study.
3. Zero-shot feature-group ablation (Sim -> Real VeHIF) to diagnose the asymmetry domain gap.
4. Leak-free few-shot real-VeHIF adaptation (5% / 10% / 20%, 5 recording-disjoint seeds, strict FA guard).

What is real vs simulated:
- REAL: VeHIF 22 kV, 50 Hz vegetation fault currents and 53 background recordings (Gomes & Ozansoy 2022).
- SIMULATED: SCT-013 + Sallen-Key + 12-bit ADC sensor front end, feeder residual load, Emanuel arcs,
  and 7 normal grid disturbances. Not a field test of physical hardware.
"""
import json
import os
import subprocess
import sys
import time
from fractions import Fraction
from pathlib import Path

import matplotlib
if "ipykernel" not in sys.modules:
    matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
from scipy import signal
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler

from hif.features import FEATURES, K, window_features, window_labels
from hif.frontend import (
    F_LINE, F_LINE_VN, FS_SIM, N_CYCLE, OVERSAMPLE, STAGES, STAGES_50HZ_RETUNED,
    codes_to_amps, frequency_response_summary, sense, sim_rate_hz,
)
from hif.waveforms import DISTURBANCES, SURFACES, WARMUP, _base, make_event
from real_data_test import (
    BINS, GRID_HZ, attr, by_bins, cycle_ms, find_h5, load_model,
    read_head, read_tail, rolling_rms, samples_per_cycle,
)
from train_kaggle import (
    HIDDEN, MAX_FALSE_ALARM, N_EVENTS, PERSIST, SEED,
    alarms, event_result, fold_and_quantize, pick_threshold, predict_q, split, stack, summarize,
)

OUT = Path("results")
FIG = OUT / "figures"
FW = Path("firmware")
BUILD = Path("build")
for d in (OUT, FIG, FW, BUILD):
    d.mkdir(parents=True, exist_ok=True)

N_VEHIF_POOL = 300         # 250 Phase-1 recordings + 50 additional valid recordings
N_PHASE1_COMPARE = 250     # exact Phase-1 seed=7 subset for 50 Hz vs 60 Hz & ablation
MAX_SECONDS = 90           # last 90 s of each recording (keeps runtime ~12-18 min on Kaggle CPU)
MAX_MINUTES = 45
VEHIF_SEED = 7
ADAPT_SEEDS = [0, 1, 2, 3, 4]
ADAPT_FRACS = [0.05, 0.10, 0.20]
SPC = N_CYCLE * OVERSAMPLE  # 2048 samples/cycle

ABLATION_SETS = {
    "all_16": list(FEATURES),
    "no_asymmetry_12": [f for f in FEATURES if f not in ("dc_ratio", "h2_ratio", "even_ratio", "even_std")],
    "no_top_shift_9": [f for f in FEATURES if f not in (
        "dc_ratio", "h2_ratio", "even_ratio", "even_std", "I3_cv", "ph3_jitter", "I1_cv")],
    "harmonics_only_7": ["log_I1", "log_I3", "h2_ratio", "h3_ratio", "h5_ratio", "even_ratio", "dc_ratio"],
    "temporal_only_8": ["I3_cv", "I1_cv", "ph3_jitter", "cycle_diff", "even_std", "h3_trend", "I1_trend", "interharm"],
}


def save_fig(fig, name):
    fig.tight_layout()
    fig.savefig(FIG / name, dpi=130)
    plt.close(fig)


# ---------------------------------------------------------------- 1. C firmware GCC check
def verify_c_firmware(header_name, vectors_path, tag):
    """Compile firmware/hif_infer.c + test_infer.c with gcc -O2 -Wall -Wextra -Werror and run."""
    bin_path = BUILD / f"test_infer_{tag}"
    cmd = [
        "gcc", "-O2", "-Wall", "-Wextra", "-Werror",
        "-I", str(FW),
        f'-DHIF_MODEL_HEADER="{header_name}"',
        str(FW / "hif_infer.c"), str(FW / "test_infer.c"),
        "-lm", "-o", str(bin_path),
    ]
    comp = subprocess.run(cmd, capture_output=True, text=True)
    if comp.returncode != 0:
        return {"compiled": False, "returncode": comp.returncode,
                "stderr": comp.stderr[-1000:], "header": header_name}
    ver = subprocess.run(["gcc", "--version"], capture_output=True, text=True)
    gcc_line = ver.stdout.splitlines()[0].strip() if ver.stdout else "gcc"
    run = subprocess.run([str(bin_path), str(vectors_path)], capture_output=True, text=True)
    res = json.loads(run.stdout) if run.stdout.strip().startswith("{") else {"raw_stdout": run.stdout}
    res.update({
        "compiled": True,
        "compiler": gcc_line,
        "compile_flags": "-O2 -Wall -Wextra -Werror",
        "header": header_name,
        "binary_bytes": bin_path.stat().st_size,
        "exit_code": run.returncode,
    })
    return res


def write_extended_vectors(layers, X_pool, y_pool, out_path, n_vecs=100, seed=42):
    rng = np.random.default_rng(seed)
    p = predict_q(layers, X_pool)
    # Stratified sample across low, mid (near decision boundary), and high probability
    low = np.flatnonzero(p < 0.05)
    mid = np.flatnonzero((p >= 0.05) & (p <= 0.995))
    hi = np.flatnonzero(p > 0.995)
    picks = []
    for arr, k in ((low, n_vecs // 3), (mid, n_vecs // 3), (hi, n_vecs - 2 * (n_vecs // 3))):
        if len(arr):
            picks.append(rng.choice(arr, size=min(k, len(arr)), replace=False))
    idx = np.concatenate(picks) if picks else np.arange(min(n_vecs, len(X_pool)))
    vecs = [{"features": [float(v) for v in X_pool[i]], "prob": float(p[i]), "is_hif": bool(y_pool[i] == 1)}
            for i in idx]
    out_path.write_text(json.dumps(vecs, indent=2), encoding="utf-8")
    return out_path


def export_c_header(layers, thr, header_path, comment):
    def carr(a, fmt):
        return "{" + ", ".join(fmt(v) for v in np.ravel(a)) + "}"
    def carr2d(a2d, fmt):
        return "{" + ", ".join("{" + ", ".join(fmt(v) for v in row) + "}" for row in a2d) + "}"
    lines = [
        f"// {comment}",
        "#pragma once",
        "#include <stdint.h>",
        f"#define HIF_N_IN {layers[0]['Wq'].shape[0]}",
        f"#define HIF_PERSIST {PERSIST}",
        f"static const float HIF_THRESHOLD = {thr:.6f}f;",
    ]
    for n, L in enumerate(layers, 1):
        Wt = L["Wq"].T
        lines += [
            f"#define HIF_L{n}_OUT {Wt.shape[0]}",
            f"#define HIF_L{n}_IN {Wt.shape[1]}",
            f"static const int8_t HIF_W{n}[{Wt.shape[0]}][{Wt.shape[1]}] = " + carr2d(Wt, lambda v: str(int(v))) + ";",
            f"static const float HIF_S{n} = {L['scale']:.8e}f;",
            f"static const float HIF_B{n}[{Wt.shape[0]}] = " + carr(L["b"], lambda v: f"{v:.8e}f") + ";",
        ]
    header_path.write_text("\n".join(lines) + "\n", encoding="utf-8")


# ---------------------------------------------------------------- 2. Fast VeHIF extraction
def to_sensor_rate_fast(x, spc):
    """Resample so 1 power cycle = 2048 samples with bounded polyphase denominator (pitch error < 0.005%)."""
    fr = Fraction(SPC / spc).limit_denominator(256)
    return signal.resample_poly(x, fr.numerator, fr.denominator)


def through_sensor_freq(x, rng, f_line):
    amps = codes_to_amps(sense(x, rng, f_line=f_line))[WARMUP * N_CYCLE:]
    X, _ = window_features(amps)
    ends = np.arange(K - 1, K - 1 + len(X)) + WARMUP
    return X, ends


def extract_vehif_record(g, rec_id, seed_pair, also_60hz=False):
    spc = samples_per_cycle(read_head(g["voltage_lf"], 1 << 18)) if "voltage_lf" in g else 2000.0
    raw, n_total = read_tail(g["current_lf"], int(MAX_SECONDS * GRID_HZ * spc))
    raw = np.nan_to_num(raw)
    x = to_sensor_rate_fast(raw - raw.mean(), spc)
    n = len(x) // SPC * SPC
    if n < 40 * SPC:
        raise ValueError("recording shorter than 40 cycles")
    x = x[:n]
    ms = cycle_ms(x)
    r8 = rolling_rms(ms, K)
    peak_rms20 = float(rolling_rms(ms, 20).max())

    # Native 50 Hz (Vietnam / Australia VeHIF physical rate: FS_ADC = 12,800 Hz)
    rng50 = np.random.default_rng(seed_pair)
    t50 = np.arange(n) / sim_rate_hz(F_LINE_VN)
    load50, a1_50, _ = _base(t50, rng50, w=2 * np.pi * F_LINE_VN)
    Xf50, ends = through_sensor_freq(load50 + x, rng50, f_line=F_LINE_VN)
    Xc50, _ = through_sensor_freq(load50, rng50, f_line=F_LINE_VN)
    win_r8 = r8[ends - (K - 1)].astype(np.float32)

    rec = {
        "id": rec_id,
        "fault_type": attr(g, "fault_type", "test_type", "cal_type") or "?",
        "seconds": float(n_total / spc / GRID_HZ),
        "peak_rms20": peak_rms20,
        "rms_all": float(np.sqrt(ms.mean())),
        "load_rms": float(a1_50 / np.sqrt(2)),
        "ratio": float(peak_rms20 / (a1_50 / np.sqrt(2))),
        "Xf50": Xf50,
        "Xc50": Xc50,
        "r8": win_r8,
    }
    if also_60hz:
        # Paired 60 Hz resampled mode (matches Phase 1 real_data_test.py rate: FS_ADC = 15,360 Hz)
        rng60 = np.random.default_rng(seed_pair)
        t60 = np.arange(n) / FS_SIM
        load60, _, _ = _base(t60, rng60, w=2 * np.pi * F_LINE)
        Xf60, _ = through_sensor_freq(load60 + x, rng60, f_line=F_LINE)
        Xc60, _ = through_sensor_freq(load60, rng60, f_line=F_LINE)
        rec["Xf60"] = Xf60
        rec["Xc60"] = Xc60
    return rec


def eval_vehif_records(records, layers, thr, feat_idx=None, mode="50hz"):
    xf_key = "Xf50" if mode == "50hz" else "Xf60"
    xc_key = "Xc50" if mode == "50hz" else "Xc60"
    rows = []
    for r in records:
        Xf = r[xf_key] if feat_idx is None else r[xf_key][:, feat_idx]
        Xc = r[xc_key] if feat_idx is None else r[xc_key][:, feat_idx]
        sf = predict_q(layers, Xf)
        sc = predict_q(layers, Xc)
        af = alarms(sf, thr)
        ac = alarms(sc, thr)
        alarm_f, alarm_c = bool(af), bool(ac)
        rows.append({
            "id": r["id"],
            "peak_rms20": r["peak_rms20"],
            "ratio": r["ratio"],
            "fault_type": r["fault_type"],
            "alarm": alarm_f,
            "control_alarm": alarm_c,
            "detected": alarm_f and not alarm_c,
        })
    low02 = [row["detected"] for row in rows if row["peak_rms20"] < 2.0]
    return {
        "recordings": len(rows),
        "detection_rate": float(np.mean([row["detected"] for row in rows])),
        "detection_rate_0_to_2A": float(np.mean(low02)) if low02 else None,
        "n_0_to_2A": len(low02),
        "alarm_in_control_rate": float(np.mean([row["control_alarm"] for row in rows])),
        "by_peak_fault_current_A_rms": by_bins(rows, "peak_rms20", BINS),
    }


def eval_bg_records(bg_records, layers, thr, feat_idx=None, mode="50hz"):
    xf_key = "Xf50" if mode == "50hz" else "Xf60"
    xc_key = "Xc50" if mode == "50hz" else "Xc60"
    al_f, al_c = [], []
    for r in bg_records:
        Xf = r[xf_key] if feat_idx is None else r[xf_key][:, feat_idx]
        Xc = r[xc_key] if feat_idx is None else r[xc_key][:, feat_idx]
        al_f.append(bool(alarms(predict_q(layers, Xf), thr)))
        al_c.append(bool(alarms(predict_q(layers, Xc), thr)))
    return {
        "recordings": len(bg_records),
        "alarm_rate": float(np.mean(al_f)) if al_f else 0.0,
        "alarm_in_control_rate": float(np.mean(al_c)) if al_c else 0.0,
    }


# ---------------------------------------------------------------- 3. Training & adaptation helpers
def simulate_phase0_events(f_line=F_LINE, n_events=N_EVENTS, seed=SEED):
    rng = np.random.default_rng(seed)
    events = []
    for k in range(n_events):
        amps, meta = make_event(rng, is_hif=(k % 2 == 0), f_line=f_line)
        X, ends = window_features(amps)
        events.append({"X": X, "ends": ends, "y": window_labels(ends, meta), **meta})
    return events


def fit_subset_int8(tr, va, feat_idx, seed=SEED):
    X_tr, y_tr = stack(tr)
    X_tr = X_tr[:, feat_idx]
    sc = StandardScaler().fit(X_tr)
    clf = MLPClassifier(hidden_layer_sizes=HIDDEN, alpha=1e-3, max_iter=400,
                        early_stopping=True, n_iter_no_change=15, random_state=seed)
    clf.fit(sc.transform(X_tr), y_tr)
    layers = fold_and_quantize(sc, clf)
    grid = np.linspace(0.30, 0.999, 300)
    va_scores = [predict_q(layers, e["X"][:, feat_idx]) for e in va]
    thr = pick_threshold(va, va_scores, grid)
    return layers, thr


def pick_joint_threshold(sim_va, sim_va_scores, real_ctrl_scores, max_sim_fa=MAX_FALSE_ALARM, max_real_fa=0.02):
    """Pick lowest threshold satisfying BOTH simulated val FA <= max_sim_fa AND real control FA <= max_real_fa."""
    grid = np.linspace(0.30, 0.999, 350)
    def ok(thr):
        s_fa = float(np.mean([event_result(e, s, thr)["false_alarm"] for e, s in zip(sim_va, sim_va_scores)]))
        if s_fa > max_sim_fa:
            return False
        if real_ctrl_scores:
            r_fa = float(np.mean([bool(alarms(s, thr)) for s in real_ctrl_scores]))
            if r_fa > max_real_fa:
                return False
        return True

    lo, hi = 0, len(grid) - 1
    if not ok(grid[hi]):
        return float(grid[hi])
    while lo < hi:
        mid = (lo + hi) // 2
        if ok(grid[mid]):
            hi = mid
        else:
            lo = mid + 1
    return float(grid[lo])


def sample_real_adapt_windows(adapt_recs, bg_adapt_recs, rng, max_per_rec=220):
    """Extract active fault windows (y=1) and paired control/background windows (y=0) from adaptation pool."""
    Xs, ys = [], []
    for r in adapt_recs:
        r8 = r["r8"]
        thresh_i = min(0.20, max(0.08, 0.45 * r["peak_rms20"]))
        pos_idx = np.flatnonzero(r8 >= thresh_i)
        if len(pos_idx) == 0:
            pos_idx = np.array([int(np.argmax(r8))])
        if len(pos_idx) > max_per_rec:
            pos_idx = np.sort(rng.choice(pos_idx, size=max_per_rec, replace=False))
        Xf_pos = r["Xf50"][pos_idx]

        n_ctrl = min(len(r["Xc50"]), max_per_rec)
        ctrl_idx = np.sort(rng.choice(len(r["Xc50"]), size=n_ctrl, replace=False))
        Xc_neg = r["Xc50"][ctrl_idx]

        Xs.append(Xf_pos)
        ys.append(np.ones(len(Xf_pos), dtype=np.int8))
        Xs.append(Xc_neg)
        ys.append(np.zeros(len(Xc_neg), dtype=np.int8))

    for br in bg_adapt_recs:
        n_bg = min(len(br["Xf50"]), 100)
        idx = np.sort(rng.choice(len(br["Xf50"]), size=n_bg, replace=False))
        Xs.append(br["Xf50"][idx])
        ys.append(np.zeros(n_bg, dtype=np.int8))
    return np.concatenate(Xs, axis=0), np.concatenate(ys, axis=0)


def hidden_repr(layers, X):
    """Run frozen Layer 1 & Layer 2 (ReLU) to extract 8-dim penultimate representation."""
    h = X.astype(np.float32)
    for L in layers[:2]:
        h = np.maximum(h @ (L["Wq"].astype(np.float32) * L["scale"]) + L["b"], 0.0)
    return h


def fit_head_recal_int8(phase0_layers, X_sim, y_sim, X_real, y_real, seed):
    """Freeze Layer 1-2 of Phase 0 int8 MLP; recalibrate Layer 3 (9 params) on Sim + Real; keep 504 B int8."""
    rng = np.random.default_rng(seed)
    n_sim_sub = min(len(X_sim), max(12000, len(X_real) * 3))
    sub = rng.choice(len(X_sim), size=n_sim_sub, replace=False)
    H_sim = hidden_repr(phase0_layers, X_sim[sub])
    H_real = hidden_repr(phase0_layers, X_real)
    H = np.concatenate([H_sim, H_real], axis=0)
    y = np.concatenate([y_sim[sub], y_real], axis=0)
    # Weight real adaptation windows to carry ~30% total weight
    w_real = (0.30 / 0.70) * (len(H_sim) / max(1, len(H_real)))
    sw = np.concatenate([np.ones(len(H_sim)), np.full(len(H_real), w_real)])
    lr = LogisticRegression(C=1.0, max_iter=500, random_state=seed)
    lr.fit(H, y, sample_weight=sw)
    W3 = lr.coef_.T.astype(np.float32)      # (8, 1)
    b3 = lr.intercept_.astype(np.float32)   # (1,)
    s3 = float(np.abs(W3).max() / 127.0) or 1e-6
    new_layers = [
        {"Wq": phase0_layers[0]["Wq"].copy(), "scale": phase0_layers[0]["scale"], "b": phase0_layers[0]["b"].copy()},
        {"Wq": phase0_layers[1]["Wq"].copy(), "scale": phase0_layers[1]["scale"], "b": phase0_layers[1]["b"].copy()},
        {"Wq": np.round(W3 / s3).astype(np.int8), "scale": s3, "b": b3},
    ]
    return new_layers


def fit_joint_mlp_int8(X_sim, y_sim, X_real, y_real, seed):
    """Jointly train a 504 B int8 (16, 8) Tiny MLP on Sim + few-shot Real VeHIF (oversampled ~25% mix)."""
    rng = np.random.default_rng(seed)
    # Repeat real windows so real takes ~25% of training samples (MLPClassifier has no sample_weight in all sklearn versions)
    target_real = int(0.33 * len(X_sim))
    reps = max(1, int(np.ceil(target_real / max(1, len(X_real)))))
    idx_real = np.tile(np.arange(len(X_real)), reps)[:target_real]
    rng.shuffle(idx_real)
    X_all = np.concatenate([X_sim, X_real[idx_real]], axis=0)
    y_all = np.concatenate([y_sim, y_real[idx_real]], axis=0)
    perm = rng.permutation(len(X_all))
    X_all, y_all = X_all[perm], y_all[perm]

    sc = StandardScaler().fit(X_all)
    clf = MLPClassifier(hidden_layer_sizes=HIDDEN, alpha=1e-3, max_iter=350,
                        early_stopping=True, n_iter_no_change=12, random_state=seed)
    clf.fit(sc.transform(X_all), y_all)
    return fold_and_quantize(sc, clf)


def stratified_split_indices(records, test_frac, seed):
    """Recording-disjoint stratified split by peak fault current bin."""
    rng = np.random.default_rng(seed)
    bins = {}
    for idx, r in enumerate(records):
        v = r["peak_rms20"]
        b = 0 if v < 1.0 else (1 if v < 2.0 else (2 if v < 5.0 else 3))
        bins.setdefault(b, []).append(idx)
    test_idx, pool_idx = [], []
    for b in sorted(bins):
        arr = np.array(bins[b], dtype=int)
        rng.shuffle(arr)
        n_te = int(round(len(arr) * test_frac))
        test_idx.extend(arr[:n_te].tolist())
        pool_idx.extend(arr[n_te:].tolist())
    rng.shuffle(pool_idx)
    return test_idx, pool_idx


def stratified_prefix(records, pool_idx, n_take, seed):
    """Take n_take recordings from pool_idx stratified by peak fault current bin."""
    rng = np.random.default_rng(seed)
    bins = {}
    for idx in pool_idx:
        v = records[idx]["peak_rms20"]
        b = 0 if v < 1.0 else (1 if v < 2.0 else (2 if v < 5.0 else 3))
        bins.setdefault(b, []).append(idx)
    chosen = []
    total = len(pool_idx)
    for b in sorted(bins):
        arr = np.array(bins[b], dtype=int)
        rng.shuffle(arr)
        k = max(1, int(round(n_take * len(arr) / total))) if len(arr) else 0
        chosen.extend(arr[:k].tolist())
    # Adjust exact count
    rem = [i for i in pool_idx if i not in set(chosen)]
    rng.shuffle(rem)
    while len(chosen) < n_take and rem:
        chosen.append(rem.pop())
    return chosen[:n_take]


# ---------------------------------------------------------------- 4. Figures
def make_figures(res):
    # Figure 1: Few-shot adaptation curve (overall + 0-2 A + sim retention + FA)
    adapt = res["few_shot_adaptation"]
    fracs_pct = [0, 5, 10, 20]
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10.5, 4.0))

    z0 = adapt["zero_shot_phase0"]
    for method, label, color, marker in (
        ("head_recal_int8", "Head recalibration (frozen trunk, 504 B)", "#0284c7", "s"),
        ("joint_mlp_int8", "Joint fine-tuned Tiny MLP (504 B int8)", "#16a34a", "o"),
    ):
        ys_all = [z0["real_detection_mean"]] + [adapt["by_fraction"][str(f)][method]["real_detection_mean"] for f in ADAPT_FRACS]
        es_all = [z0["real_detection_std"]] + [adapt["by_fraction"][str(f)][method]["real_detection_std"] for f in ADAPT_FRACS]
        ys_low = [z0["real_0_to_2A_mean"]] + [adapt["by_fraction"][str(f)][method]["real_0_to_2A_mean"] for f in ADAPT_FRACS]
        es_low = [z0["real_0_to_2A_std"]] + [adapt["by_fraction"][str(f)][method]["real_0_to_2A_std"] for f in ADAPT_FRACS]
        ax1.errorbar(fracs_pct, [v * 100 for v in ys_all], yerr=[v * 100 for v in es_all],
                     marker=marker, lw=2, capsize=4, color=color, label=label + " — All")
        ax1.errorbar(fracs_pct, [v * 100 for v in ys_low], yerr=[v * 100 for v in es_low],
                     marker=marker, lw=1.6, ls="--", capsize=3, color=color, alpha=0.8, label=label + " — 0–2 A")
    ax1.set_xlabel("Real VeHIF recordings used for adaptation (% of 300 pool)")
    ax1.set_ylabel("Held-out real VeHIF detection (%)")
    ax1.set_ylim(0, 105)
    ax1.set_xticks(fracs_pct)
    ax1.grid(alpha=0.25)
    ax1.legend(fontsize=7.5, loc="lower right")
    ax1.set_title("Few-shot adaptation on unseen real VeHIF 22 kV (5 seeds)")

    # Right panel: Band-by-band comparison (Zero-shot vs 20% Joint MLP int8)
    bands = ["0-1", "1-2", "2-5", "5-10"]
    x = np.arange(len(bands))
    w = 0.36
    z_bands = [(z0["by_current_band_mean"].get(b) or 0.0) * 100 for b in bands]
    j20 = adapt["by_fraction"]["0.2"]["joint_mlp_int8"]
    j_bands = [(j20["by_current_band_mean"].get(b) or 0.0) * 100 for b in bands]
    ax2.bar(x - w / 2, z_bands, w, color="#94a3b8", label="Phase 0 Zero-shot (504 B)")
    ax2.bar(x + w / 2, j_bands, w, color="#16a34a", label="Phase 1.5 Joint 20% (504 B)")
    for i, (zv, jv) in enumerate(zip(z_bands, j_bands)):
        ax2.text(i - w / 2, zv + 1.8, f"{zv:.0f}%", ha="center", fontsize=8)
        ax2.text(i + w / 2, jv + 1.8, f"{jv:.0f}%", ha="center", fontsize=8, fontweight="bold")
    ax2.set_xticks(x, [f"{b} A" for b in bands])
    ax2.set_ylabel("Held-out detection (%)")
    ax2.set_ylim(0, 110)
    ax2.grid(axis="y", alpha=0.25)
    ax2.legend(fontsize=8)
    ax2.set_title("Detection by real fault current band (50 Hz native)")
    save_fig(fig, "phase15_adaptation.png")

    # Figure 2: Feature-group ablation (Sim vs Zero-shot Real VeHIF)
    abl = res["feature_ablation"]
    names = list(ABLATION_SETS.keys())
    labels = ["All 16\n(Phase 0)", "No Asym\n(12)", "No Top-7\nShift (9)", "Harmonics\nOnly (7)", "Temporal\nOnly (8)"]
    sim_det = [abl[n]["sim_test"]["detection_rate"] * 100 for n in names]
    real_det = [abl[n]["real_vehif_50hz"]["detection_rate"] * 100 for n in names]
    sim_fa = [abl[n]["sim_test"]["false_alarm_rate_normal"] * 100 for n in names]
    fig, ax = plt.subplots(figsize=(8.6, 3.8))
    x = np.arange(len(names))
    w = 0.28
    ax.bar(x - w, sim_det, w, color="#2563eb", label="Simulated Emanuel test detection (%)")
    ax.bar(x, real_det, w, color="#ea580c", label="Real VeHIF 22 kV zero-shot detection (%)")
    ax.bar(x + w, sim_fa, w, color="#dc2626", label="Simulated normal false alarms (%)")
    for i in range(len(names)):
        ax.text(i - w, sim_det[i] + 1.5, f"{sim_det[i]:.1f}", ha="center", fontsize=7.5)
        ax.text(i, real_det[i] + 1.5, f"{real_det[i]:.1f}", ha="center", fontsize=7.5)
        ax.text(i + w, sim_fa[i] + 1.5, f"{sim_fa[i]:.1f}", ha="center", fontsize=7.5)
    ax.set_xticks(x, labels)
    ax.set_ylim(0, 112)
    ax.set_ylabel("Rate (%)")
    ax.grid(axis="y", alpha=0.25)
    ax.legend(fontsize=8)
    ax.set_title("Feature-group ablation: simulated accuracy vs zero-shot real VeHIF transfer")
    save_fig(fig, "phase15_ablation.png")

    # Figure 3: 50 Hz Vietnam vs 60 Hz US front-end Bode & VeHIF evaluation
    f50 = res["freq_50hz_vs_60hz"]
    freqs = np.logspace(1.4, 4.3, 1200)
    _, h_std = signal.freqs([1.0], np.polymul(
        [STAGES[0][0] ** 2 * STAGES[0][1] * STAGES[0][2], 2 * STAGES[0][0] * STAGES[0][2], 1.0],
        [STAGES[1][0] ** 2 * STAGES[1][1] * STAGES[1][2], 2 * STAGES[1][0] * STAGES[1][2], 1.0],
    ), worN=2 * np.pi * freqs)
    _, h_50r = signal.freqs([1.0], np.polymul(
        [STAGES_50HZ_RETUNED[0][0] ** 2 * STAGES_50HZ_RETUNED[0][1] * STAGES_50HZ_RETUNED[0][2],
         2 * STAGES_50HZ_RETUNED[0][0] * STAGES_50HZ_RETUNED[0][2], 1.0],
        [STAGES_50HZ_RETUNED[1][0] ** 2 * STAGES_50HZ_RETUNED[1][1] * STAGES_50HZ_RETUNED[1][2],
         2 * STAGES_50HZ_RETUNED[1][0] * STAGES_50HZ_RETUNED[1][2], 1.0],
    ), worN=2 * np.pi * freqs)
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(10.4, 3.8))
    a1.semilogx(freqs, 20 * np.log10(np.abs(h_std)), lw=2, color="#2563eb", label="Current PCB (fc = 2,048 Hz)")
    a1.semilogx(freqs, 20 * np.log10(np.abs(h_50r)), lw=1.8, ls="--", color="#16a34a", label="50 Hz Retuned R×1.2 (fc ≈ 1,688 Hz)")
    a1.axvline(6400, color="#ea580c", ls=":", lw=1.3, label="50 Hz Nyquist (6.40 kHz)")
    a1.axvline(7680, color="#64748b", ls=":", lw=1.3, label="60 Hz Nyquist (7.68 kHz)")
    a1.set_ylim(-85, 5)
    a1.set_xlabel("Frequency (Hz)")
    a1.set_ylabel("Filter gain (dB)")
    a1.grid(alpha=0.25)
    a1.legend(fontsize=7.5)
    a1.set_title("Same PCB anti-aliasing margin: 50 Hz (VN) vs 60 Hz (US)")

    bands = ["0-1", "1-2", "2-5", "5-10"]
    x = np.arange(len(bands))
    w = 0.36
    b60 = [((f50["vehif_250_resampled_60hz"]["by_peak_fault_current_A_rms"].get(b) or {}).get("detected") or 0.0) * 100 for b in bands]
    b50 = [((f50["vehif_250_native_50hz"]["by_peak_fault_current_A_rms"].get(b) or {}).get("detected") or 0.0) * 100 for b in bands]
    a2.bar(x - w / 2, b60, w, color="#2563eb", label=f"60 Hz resampled ({f50['vehif_250_resampled_60hz']['detection_rate']*100:.1f}%)")
    a2.bar(x + w / 2, b50, w, color="#ea580c", label=f"50 Hz native VN/AU ({f50['vehif_250_native_50hz']['detection_rate']*100:.1f}%)")
    a2.set_xticks(x, [f"{b} A" for b in bands])
    a2.set_ylabel("VeHIF zero-shot detection (%)")
    a2.set_ylim(0, 108)
    a2.grid(axis="y", alpha=0.25)
    a2.legend(fontsize=8)
    a2.set_title("Phase 0 frozen model on 250 VeHIF: 50 Hz vs 60 Hz")
    save_fig(fig, "phase15_50hz_vs_60hz.png")


# ---------------------------------------------------------------- 5. Main runner
def main():
    t_start = time.time()
    print("=== PHASE 1.5: C Firmware Check + 50/60 Hz + Feature Ablation + VeHIF Few-Shot Adaptation ===")

    if not (OUT / "model_int8.npz").exists():
        raise FileNotFoundError("results/model_int8.npz missing (attach Phase 0 model first)")
    phase0_layers, phase0_thr, _ = load_model(OUT / "model_int8.npz")

    # 1. Verify Phase 0 C firmware on official test_vectors.json right away
    print("\n[1/5] Compiling & verifying Phase 0 C firmware (firmware/hif_infer.c + firmware/hif_model.h)...")
    c_phase0_official = verify_c_firmware("hif_model.h", OUT / "test_vectors.json", "phase0_official")
    print("  Phase 0 official 6 vectors:", json.dumps(c_phase0_official))

    # 2. Simulate Phase 0 events (exact same seed=42 & 60/20/20 split as train_kaggle.py)
    print("\n[2/5] Simulating Phase 0 60 Hz reference events (N=4000, seed=42)...")
    t_sim = time.time()
    sim_events = simulate_phase0_events(f_line=F_LINE, n_events=N_EVENTS, seed=SEED)
    sim_tr, sim_va, sim_te = split(sim_events, np.random.default_rng(SEED))
    X_sim_tr, y_sim_tr = stack(sim_tr)
    X_sim_te, y_sim_te = stack(sim_te)
    print(f"  Simulated {len(sim_events)} events in {time.time() - t_sim:.1f} s (tr/va/te = {len(sim_tr)}/{len(sim_va)}/{len(sim_te)})")

    # Extended 100-vector C firmware verification for Phase 0
    ext_vec_p0 = write_extended_vectors(phase0_layers, X_sim_te, y_sim_te, BUILD / "test_vectors_phase0_100.json", n_vecs=100, seed=42)
    c_phase0_ext = verify_c_firmware("hif_model.h", ext_vec_p0, "phase0_ext100")
    print(f"  Phase 0 extended 100 vectors: max_err={c_phase0_ext.get('max_abs_prob_error')}, "
          f"agree={c_phase0_ext.get('decision_agreement')}, lat={c_phase0_ext.get('host_latency_us_per_inference')} us")

    # Also test simulated 50 Hz (Vietnam) grid events (N=600) with Phase 0 model
    print("  Simulating 600 native 50 Hz (Vietnam) events to test 50 Hz simulated transfer...")
    sim50_events = simulate_phase0_events(f_line=F_LINE_VN, n_events=600, seed=SEED + 50)
    sim50_summary = summarize(sim50_events, [predict_q(phase0_layers, e["X"]) for e in sim50_events], phase0_thr)
    sim60_summary = summarize(sim_te, [predict_q(phase0_layers, e["X"]) for e in sim_te], phase0_thr)

    # 3. Read real VeHIF 22 kV recordings (250 exact Phase-1 subset + 50 extra = 300, plus 53 background)
    h5_path = find_h5()
    if not h5_path:
        raise FileNotFoundError("VeHIF HDF5 dataset not found under /kaggle/input/ or VEHIF_PATH")
    import h5py
    print(f"\n[3/5] Extracting {N_VEHIF_POOL} real VeHIF fault recordings + 53 background from {h5_path}...")
    t_ve = time.time()
    fault_recs, bg_recs, errors = [], [], []
    with h5py.File(h5_path, "r") as f:
        tg = f[next(k for k in f.keys() if k.lower().startswith("test"))]
        cg = f[next(k for k in f.keys() if k.lower().startswith("cal"))]
        ids = sorted(tg.keys())
        valid = [i for i in ids if (attr(tg[i], "report_validity") or "valid").lower() != "invalid"]
        order = np.random.default_rng(VEHIF_SEED).permutation(len(valid))
        p1_ids = sorted(valid[k] for k in order[:N_PHASE1_COMPARE])
        p1_set = set(p1_ids)
        extra_ids = [valid[k] for k in order[N_PHASE1_COMPARE:N_VEHIF_POOL]]
        chosen = p1_ids + extra_ids

        for idx, rec_id in enumerate(chosen):
            if (time.time() - t_ve) / 60.0 > MAX_MINUTES:
                print(f"  time budget reached at {idx}/{len(chosen)} recordings")
                break
            try:
                r = extract_vehif_record(tg[rec_id], rec_id, [VEHIF_SEED, idx], also_60hz=(rec_id in p1_set))
                r["in_phase1_250"] = rec_id in p1_set
                fault_recs.append(r)
            except Exception as e:
                errors.append(f"test {rec_id}: {type(e).__name__}: {e}"[:140])
            if (idx + 1) % 50 == 0:
                print(f"  extracted {idx + 1}/{len(chosen)} fault recordings ({time.time() - t_ve:.0f} s)")

        for idx, rec_id in enumerate(sorted(cg.keys())):
            try:
                bg_recs.append(extract_vehif_record(cg[rec_id], rec_id, [VEHIF_SEED, 100000 + idx], also_60hz=True))
            except Exception as e:
                errors.append(f"bg {rec_id}: {type(e).__name__}: {e}"[:140])
    print(f"  Extracted {len(fault_recs)} fault + {len(bg_recs)} background recordings in {time.time() - t_ve:.1f} s")

    p1_recs = [r for r in fault_recs if r["in_phase1_250"]]

    # 4. 50 Hz (Vietnam) vs 60 Hz (US) evaluation
    print("\n[4/5] Running 50 Hz vs 60 Hz comparison & Feature-Group Ablation...")
    freq_comp = {
        "hardware_60hz_us": frequency_response_summary(F_LINE, STAGES),
        "hardware_50hz_vn_same_pcb": frequency_response_summary(F_LINE_VN, STAGES),
        "hardware_50hz_vn_retuned_R1p2": frequency_response_summary(F_LINE_VN, STAGES_50HZ_RETUNED),
        "sim_60hz_test_800": {
            "detection_rate": sim60_summary["detection_rate"],
            "false_alarm_rate_normal": sim60_summary["false_alarm_rate_normal"],
        },
        "sim_50hz_test_600": {
            "detection_rate": sim50_summary["detection_rate"],
            "false_alarm_rate_normal": sim50_summary["false_alarm_rate_normal"],
        },
        "vehif_250_resampled_60hz": eval_vehif_records(p1_recs, phase0_layers, phase0_thr, mode="60hz"),
        "vehif_250_native_50hz": eval_vehif_records(p1_recs, phase0_layers, phase0_thr, mode="50hz"),
        "bg_53_resampled_60hz": eval_bg_records(bg_recs, phase0_layers, phase0_thr, mode="60hz"),
        "bg_53_native_50hz": eval_bg_records(bg_recs, phase0_layers, phase0_thr, mode="50hz"),
    }
    print(f"  250 VeHIF zero-shot: 60Hz={freq_comp['vehif_250_resampled_60hz']['detection_rate']:.1%} "
          f"vs 50Hz={freq_comp['vehif_250_native_50hz']['detection_rate']:.1%}")

    # Feature-group ablation (trained ONLY on Phase 0 simulated data, zero-shot to VeHIF 50 Hz)
    ablation_res = {}
    for name, cols in ABLATION_SETS.items():
        fidx = [FEATURES.index(c) for c in cols]
        L_ab, thr_ab = fit_subset_int8(sim_tr, sim_va, fidx, seed=SEED)
        sim_r = summarize(sim_te, [predict_q(L_ab, e["X"][:, fidx]) for e in sim_te], thr_ab)
        ve_r = eval_vehif_records(p1_recs, L_ab, thr_ab, feat_idx=fidx, mode="50hz")
        bg_r = eval_bg_records(bg_recs, L_ab, thr_ab, feat_idx=fidx, mode="50hz")
        ablation_res[name] = {
            "n_features": len(cols),
            "features": cols,
            "threshold": thr_ab,
            "sim_test": {
                "detection_rate": sim_r["detection_rate"],
                "false_alarm_rate_normal": sim_r["false_alarm_rate_normal"],
                "fa_inrush": sim_r["false_alarm_by_disturbance"]["inrush"],
                "fa_motor_start": sim_r["false_alarm_by_disturbance"]["motor_start"],
                "fa_arc_welder": sim_r["false_alarm_by_disturbance"]["arc_welder"],
            },
            "real_vehif_50hz": ve_r,
            "real_bg_50hz": bg_r,
        }
        print(f"  Ablation {name:16s}: sim_det={sim_r['detection_rate']:.1%}, sim_fa={sim_r['false_alarm_rate_normal']:.1%}, "
              f"real_vehif={ve_r['detection_rate']:.1%} (0-2A={ve_r['detection_rate_0_to_2A']:.1%}), bg_fa={bg_r['alarm_rate']:.1%}")

    # 5. Few-shot real-VeHIF adaptation (strict recording-level disjoint splits, 5 seeds)
    print("\n[5/5] Running leak-free Few-Shot Real-VeHIF Adaptation (5%, 10%, 20% x 5 seeds)...")
    z0_runs = []
    by_frac = {str(fr): {"head_recal_int8": [], "joint_mlp_int8": []} for fr in ADAPT_FRACS}
    best_model_export = None

    for s in ADAPT_SEEDS:
        te_idx, pool_idx = stratified_split_indices(fault_recs, test_frac=0.60, seed=100 + s)
        real_te = [fault_recs[i] for i in te_idx]
        rng_bg = np.random.default_rng(200 + s)
        bg_perm = rng_bg.permutation(len(bg_recs))
        bg_adapt = [bg_recs[i] for i in bg_perm[:20]]
        bg_te = [bg_recs[i] for i in bg_perm[20:]]

        # Zero-shot Phase 0 on this exact seed's held-out test split
        z_ve = eval_vehif_records(real_te, phase0_layers, phase0_thr, mode="50hz")
        z_bg = eval_bg_records(bg_te, phase0_layers, phase0_thr, mode="50hz")
        z0_runs.append({
            "seed": s, "real_ve": z_ve, "real_bg": z_bg,
            "sim_det": sim60_summary["detection_rate"],
            "sim_fa": sim60_summary["false_alarm_rate_normal"],
        })

        for fr in ADAPT_FRACS:
            n_take = max(1, int(round(fr * len(fault_recs))))
            sel_idx = stratified_prefix(fault_recs, pool_idx, n_take, seed=300 + s * 10 + n_take)
            adapt_recs = [fault_recs[i] for i in sel_idx]
            X_real, y_real = sample_real_adapt_windows(adapt_recs, bg_adapt, np.random.default_rng(400 + s * 10 + n_take))

            for method in ("head_recal_int8", "joint_mlp_int8"):
                if method == "head_recal_int8":
                    layers_m = fit_head_recal_int8(phase0_layers, X_sim_tr, y_sim_tr, X_real, y_real, seed=500 + s)
                else:
                    layers_m = fit_joint_mlp_int8(X_sim_tr, y_sim_tr, X_real, y_real, seed=600 + s)

                sim_va_scores = [predict_q(layers_m, e["X"]) for e in sim_va]
                real_ctrl_scores = [predict_q(layers_m, r["Xc50"]) for r in adapt_recs] + [
                    predict_q(layers_m, b["Xf50"]) for b in bg_adapt]
                thr_m = pick_joint_threshold(sim_va, sim_va_scores, real_ctrl_scores,
                                             max_sim_fa=MAX_FALSE_ALARM, max_real_fa=0.02)
                ve_m = eval_vehif_records(real_te, layers_m, thr_m, mode="50hz")
                bg_m = eval_bg_records(bg_te, layers_m, thr_m, mode="50hz")
                sim_m = summarize(sim_te, [predict_q(layers_m, e["X"]) for e in sim_te], thr_m)
                by_frac[str(fr)][method].append({
                    "seed": s,
                    "n_adapt_recordings": len(adapt_recs),
                    "threshold": thr_m,
                    "real_ve": ve_m,
                    "real_bg": bg_m,
                    "sim_det": sim_m["detection_rate"],
                    "sim_fa": sim_m["false_alarm_rate_normal"],
                    "sim_fa_by_dist": sim_m["false_alarm_by_disturbance"],
                })
                if s == 0 and fr == 0.20 and method == "joint_mlp_int8":
                    best_model_export = (layers_m, thr_m, ve_m, bg_m, sim_m)

        print(f"  seed {s} done | 0%: {z_ve['detection_rate']:.1%} | "
              f"20% head: {by_frac['0.2']['head_recal_int8'][-1]['real_ve']['detection_rate']:.1%} | "
              f"20% joint: {by_frac['0.2']['joint_mlp_int8'][-1]['real_ve']['detection_rate']:.1%} "
              f"(0-2A={by_frac['0.2']['joint_mlp_int8'][-1]['real_ve']['detection_rate_0_to_2A']:.1%}, "
              f"sim_fa={by_frac['0.2']['joint_mlp_int8'][-1]['sim_fa']:.1%}, "
              f"bg_fa={by_frac['0.2']['joint_mlp_int8'][-1]['real_bg']['alarm_rate']:.1%})")

    def agg(runs):
        bands = ["0-1", "1-2", "2-5", "5-10"]
        band_mean = {}
        band_std = {}
        for b in bands:
            vals = [r["real_ve"]["by_peak_fault_current_A_rms"][b]["detected"]
                    for r in runs if r["real_ve"]["by_peak_fault_current_A_rms"][b]["detected"] is not None]
            band_mean[b] = float(np.mean(vals)) if vals else None
            band_std[b] = float(np.std(vals)) if vals else None
        return {
            "n_seeds": len(runs),
            "n_adapt_recordings": runs[0].get("n_adapt_recordings", 0),
            "n_heldout_fault": runs[0]["real_ve"]["recordings"],
            "n_heldout_bg": runs[0]["real_bg"]["recordings"],
            "real_detection_mean": float(np.mean([r["real_ve"]["detection_rate"] for r in runs])),
            "real_detection_std": float(np.std([r["real_ve"]["detection_rate"] for r in runs])),
            "real_0_to_2A_mean": float(np.mean([r["real_ve"]["detection_rate_0_to_2A"] for r in runs])),
            "real_0_to_2A_std": float(np.std([r["real_ve"]["detection_rate_0_to_2A"] for r in runs])),
            "real_control_fa_mean": float(np.mean([r["real_ve"]["alarm_in_control_rate"] for r in runs])),
            "real_bg_fa_mean": float(np.mean([r["real_bg"]["alarm_rate"] for r in runs])),
            "sim_detection_mean": float(np.mean([r["sim_det"] for r in runs])),
            "sim_fa_mean": float(np.mean([r["sim_fa"] for r in runs])),
            "by_current_band_mean": band_mean,
            "by_current_band_std": band_std,
            "threshold_mean": float(np.mean([r.get("threshold", phase0_thr) for r in runs])),
        }

    adapt_summary = {
        "pool_fault_recordings": len(fault_recs),
        "pool_background_recordings": len(bg_recs),
        "split_protocol": "60% held-out real test / 40% adaptation pool, recording-ID disjoint, stratified by peak current, 5 seeds",
        "fa_guard": {"sim_val_max": MAX_FALSE_ALARM, "real_adapt_control_max": 0.02},
        "zero_shot_phase0": agg(z0_runs),
        "by_fraction": {
            str(fr): {m: agg(by_frac[str(fr)][m]) for m in ("head_recal_int8", "joint_mlp_int8")}
            for fr in ADAPT_FRACS
        },
    }

    # Export Phase 1.5 adapted 504 B int8 model + C header + run GCC verification on it!
    c_phase15 = None
    if best_model_export is not None:
        L15, thr15, _, _, _ = best_model_export
        d15 = {"thr": thr15}
        for n, L in enumerate(L15):
            d15[f"W{n}"], d15[f"s{n}"], d15[f"b{n}"] = L["Wq"], L["scale"], L["b"]
        np.savez(OUT / "model_phase15_int8.npz", **d15)
        export_c_header(L15, thr15, FW / "hif_model_phase15.h",
                        "Auto-generated by phase15_vehif_adaptation.py (20% VeHIF joint fine-tuned 504 B int8 MLP).")
        vec15_path = write_extended_vectors(L15, X_sim_te, y_sim_te, OUT / "test_vectors_phase15.json", n_vecs=100, seed=15)
        c_phase15 = verify_c_firmware("hif_model_phase15.h", vec15_path, "phase15")
        print(f"\n  Phase 1.5 C firmware check: max_err={c_phase15.get('max_abs_prob_error')}, "
              f"agree={c_phase15.get('decision_agreement')}, pass={c_phase15.get('parity_pass')}")

    # Pre-registered acceptance criteria P1.5-A..D
    z0_agg = adapt_summary["zero_shot_phase0"]
    j20_agg = adapt_summary["by_fraction"]["0.2"]["joint_mlp_int8"]
    gain_02 = j20_agg["real_0_to_2A_mean"] - z0_agg["real_0_to_2A_mean"]
    criteria = {
        "P1_5_A_low_current_gain_ge_15pp": {
            "pass": bool(gain_02 >= 0.15),
            "measured_delta_0_to_2A": round(gain_02, 4),
            "zero_shot_0_to_2A": round(z0_agg["real_0_to_2A_mean"], 4),
            "adapted_20pct_0_to_2A": round(j20_agg["real_0_to_2A_mean"], 4),
        },
        "P1_5_B_real_bg_and_control_fa_le_3pct": {
            "pass": bool(j20_agg["real_bg_fa_mean"] <= 0.03 and j20_agg["real_control_fa_mean"] <= 0.03),
            "real_bg_fa_mean": round(j20_agg["real_bg_fa_mean"], 4),
            "real_control_fa_mean": round(j20_agg["real_control_fa_mean"], 4),
        },
        "P1_5_C_no_catastrophic_forgetting_sim": {
            "pass": bool(j20_agg["sim_detection_mean"] >= 0.88 and j20_agg["sim_fa_mean"] <= 0.03),
            "sim_detection_mean": round(j20_agg["sim_detection_mean"], 4),
            "sim_fa_mean": round(j20_agg["sim_fa_mean"], 4),
        },
        "P1_5_D_c_firmware_bit_parity": {
            "pass": bool(c_phase0_official.get("parity_pass") and c_phase0_ext.get("parity_pass")
                         and (c_phase15 or {}).get("parity_pass")),
            "phase0_official_max_err": c_phase0_official.get("max_abs_prob_error"),
            "phase0_ext100_max_err": c_phase0_ext.get("max_abs_prob_error"),
            "phase15_ext100_max_err": (c_phase15 or {}).get("max_abs_prob_error"),
        },
    }

    c_verify_out = {
        "note": "Compiled and executed natively on Linux GCC with -O2 -Wall -Wextra -Werror.",
        "phase0_official_6_vectors": c_phase0_official,
        "phase0_extended_100_vectors": c_phase0_ext,
        "phase15_adapted_100_vectors": c_phase15,
    }
    (OUT / "c_firmware_verify.json").write_text(json.dumps(c_verify_out, indent=2) + "\n", encoding="utf-8")

    res = {
        "note": "REAL VeHIF 22 kV 50 Hz vegetation fault currents & 53 background recordings through SIMULATED "
                "sensor + SIMULATED feeder load, combined with SIMULATED Emanuel arcs & 7 disturbances. "
                "Not a field test of physical hardware.",
        "data": "D. Gomes and C. Ozansoy, VeHIF, IEEE Trans. Power Delivery 37(6), 2022; "
                "kaggle.com/datasets/dougpsg/highimpedance-vegetation-fault-data-set (CC0)",
        "freq_50hz_vs_60hz": freq_comp,
        "feature_ablation": ablation_res,
        "few_shot_adaptation": adapt_summary,
        "c_firmware_verify": c_verify_out,
        "criteria": criteria,
        "errors": errors[:20],
        "runtime_s": round(time.time() - t_start, 1),
    }
    (OUT / "phase15_vehif.json").write_text(json.dumps(res, indent=2) + "\n", encoding="utf-8")
    make_figures(res)

    print("\n================ PHASE 1.5 SUMMARY ================")
    print("Criteria:", json.dumps(criteria, indent=2))
    print(f"Total runtime: {res['runtime_s']} s -> results/phase15_vehif.json, results/c_firmware_verify.json")
    return res


if __name__ == "__main__":
    main()


In [ ]:
%%writefile firmware/hif_infer.h
// Public C API for the HIF tiny MLP inference engine (ESP32-S3 / STM32 / host tests).
#pragma once

#ifndef HIF_MODEL_HEADER
#define HIF_MODEL_HEADER "hif_model.h"
#endif
#include HIF_MODEL_HEADER

#ifdef __cplusplus
extern "C" {
#endif

// Predict HIF probability [0, 1] from 16 window features (same order as FEATURES in hif/features.py).
float hif_predict(const float x[HIF_N_IN]);

// Call once per power cycle. Returns 1 when HIF_PERSIST consecutive windows exceed HIF_THRESHOLD.
int hif_update(const float x[HIF_N_IN]);

// Reset internal persistence counter to 0 (call between independent events or after latch reset).
void hif_reset(void);

#ifdef __cplusplus
}
#endif


In [ ]:
%%writefile firmware/hif_infer.c
// Reference inference for the HIF tiny MLP on ESP32-S3 / STM32 (Phase 1 will add ADC + feature code).
// hif_model.h is generated by train_kaggle.py: int8 weights with one float scale per layer.
#include <math.h>
#include "hif_infer.h"

static void dense(const int8_t *w, float scale, const float *b, int n_out, int n_in,
                  const float *x, float *y, int relu) {
    for (int o = 0; o < n_out; o++) {
        float acc = 0.0f;                        // features are float, so only weights are int8
        for (int i = 0; i < n_in; i++) acc += (float)w[o * n_in + i] * x[i];
        acc = acc * scale + b[o];
        y[o] = relu ? (acc > 0.0f ? acc : 0.0f) : acc;
    }
}

// x: the 16 features of the newest 8-cycle window (same order as FEATURES in hif/features.py).
float hif_predict(const float x[HIF_N_IN]) {
    float h1[HIF_L1_OUT], h2[HIF_L2_OUT], z;
    dense(&HIF_W1[0][0], HIF_S1, HIF_B1, HIF_L1_OUT, HIF_L1_IN, x, h1, 1);
    dense(&HIF_W2[0][0], HIF_S2, HIF_B2, HIF_L2_OUT, HIF_L2_IN, h1, h2, 1);
    dense(&HIF_W3[0][0], HIF_S3, HIF_B3, HIF_L3_OUT, HIF_L3_IN, h2, &z, 0);
    return 1.0f / (1.0f + expf(-z));
}

static int run = 0;

// Call once per cycle. Returns 1 when HIF_PERSIST windows in a row are above the threshold.
int hif_update(const float x[HIF_N_IN]) {
    run = (hif_predict(x) > HIF_THRESHOLD) ? run + 1 : 0;
    return run >= HIF_PERSIST;
}

void hif_reset(void) {
    run = 0;
}


In [ ]:
%%writefile firmware/test_infer.c
// Bit-parity verification & latency benchmark for firmware/hif_infer.c against Python test_vectors.json.
// Compile: gcc -O2 -Wall -Wextra -Werror -I firmware firmware/hif_infer.c firmware/test_infer.c -lm -o build/test_infer
// Run:     ./build/test_infer results/test_vectors.json
#define _POSIX_C_SOURCE 199309L
#include <math.h>
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <time.h>
#include "hif_infer.h"

#define MAX_VECTORS 512

typedef struct {
    float features[HIF_N_IN];
    double expected_prob;
    int is_hif;
} TestVec;

static int parse_vectors(const char *path, TestVec *out, int max_n) {
    FILE *f = fopen(path, "rb");
    if (!f) {
        fprintf(stderr, "error: cannot open %s\n", path);
        return -1;
    }
    fseek(f, 0, SEEK_END);
    long sz = ftell(f);
    fseek(f, 0, SEEK_SET);
    char *buf = (char *)malloc((size_t)sz + 1);
    if (!buf) {
        fclose(f);
        return -1;
    }
    size_t nread = fread(buf, 1, (size_t)sz, f);
    buf[nread] = '\0';
    fclose(f);

    int count = 0;
    const char *p = buf;
    while (count < max_n && (p = strstr(p, "\"features\"")) != NULL) {
        const char *lb = strchr(p, '[');
        if (!lb) break;
        const char *q = lb + 1;
        int ok = 1;
        for (int i = 0; i < HIF_N_IN; i++) {
            char *endp = NULL;
            double v = strtod(q, &endp);
            if (endp == q) {
                ok = 0;
                break;
            }
            out[count].features[i] = (float)v;
            q = endp;
            while (*q == ' ' || *q == '\t' || *q == '\r' || *q == '\n' || *q == ',') q++;
        }
        const char *pp = strstr(q, "\"prob\"");
        if (!ok || !pp) break;
        const char *colon = strchr(pp, ':');
        if (!colon) break;
        char *endp = NULL;
        out[count].expected_prob = strtod(colon + 1, &endp);
        const char *ph = strstr(endp, "\"is_hif\"");
        out[count].is_hif = (ph && strstr(ph, "true") && (!strstr(ph, "false") || strstr(ph, "true") < strstr(ph, "false"))) ? 1 : 0;
        count++;
        p = endp;
    }
    free(buf);
    return count;
}

static double now_sec(void) {
#if defined(CLOCK_MONOTONIC)
    struct timespec ts;
    clock_gettime(CLOCK_MONOTONIC, &ts);
    return (double)ts.tv_sec + (double)ts.tv_nsec * 1e-9;
#else
    return (double)clock() / (double)CLOCKS_PER_SEC;
#endif
}

int main(int argc, char **argv) {
    const char *vec_path = (argc > 1) ? argv[1] : "results/test_vectors.json";
    TestVec vecs[MAX_VECTORS];
    int n = parse_vectors(vec_path, vecs, MAX_VECTORS);
    if (n <= 0) {
        fprintf(stderr, "error: parsed %d vectors from %s\n", n, vec_path);
        return 2;
    }

    double max_abs_err = 0.0;
    double mean_abs_err = 0.0;
    int decision_matches = 0;
    int pos_idx = -1, neg_idx = -1;

    for (int k = 0; k < n; k++) {
        float got = hif_predict(vecs[k].features);
        double err = fabs((double)got - vecs[k].expected_prob);
        if (err > max_abs_err) max_abs_err = err;
        mean_abs_err += err;
        int c_dec = (got > HIF_THRESHOLD) ? 1 : 0;
        int py_dec = (vecs[k].expected_prob > (double)HIF_THRESHOLD) ? 1 : 0;
        if (c_dec == py_dec) decision_matches++;
        if (c_dec == 1 && pos_idx < 0) pos_idx = k;
        if (c_dec == 0 && neg_idx < 0) neg_idx = k;
    }
    mean_abs_err /= (double)n;

    // Verify persistence state machine (HIF_PERSIST consecutive alarms required, reset clears)
    int persist_ok = 1;
    if (pos_idx >= 0 && neg_idx >= 0) {
        hif_reset();
        for (int s = 0; s < HIF_PERSIST - 1; s++) {
            if (hif_update(vecs[pos_idx].features) != 0) persist_ok = 0;
        }
        if (hif_update(vecs[neg_idx].features) != 0) persist_ok = 0;  // reset on normal window
        for (int s = 0; s < HIF_PERSIST - 1; s++) {
            if (hif_update(vecs[pos_idx].features) != 0) persist_ok = 0;
        }
        if (hif_update(vecs[pos_idx].features) != 1) persist_ok = 0;  // triggers on HIF_PERSIST-th
        hif_reset();
        if (hif_update(vecs[pos_idx].features) != 0) persist_ok = 0;
    } else {
        persist_ok = -1;
    }

    // Latency benchmark (100,000 inferences)
    const int iters = 100000;
    volatile float sink = 0.0f;
    double t0 = now_sec();
    for (int it = 0; it < iters; it++) {
        sink += hif_predict(vecs[it % n].features);
    }
    double dt = now_sec() - t0;
    double us_per_inf = (dt * 1e6) / (double)iters;
    (void)sink;

    int params = HIF_L1_OUT * HIF_L1_IN + HIF_L1_OUT + HIF_L2_OUT * HIF_L2_IN + HIF_L2_OUT + HIF_L3_OUT * HIF_L3_IN + HIF_L3_OUT;
    int model_bytes = HIF_L1_OUT * HIF_L1_IN + 4 * HIF_L1_OUT + 4 +
                      HIF_L2_OUT * HIF_L2_IN + 4 * HIF_L2_OUT + 4 +
                      HIF_L3_OUT * HIF_L3_IN + 4 * HIF_L3_OUT + 4;
    int parity_pass = (max_abs_err < 1e-5 && decision_matches == n && persist_ok == 1);

    printf("{\n");
    printf("  \"vectors_path\": \"%s\",\n", vec_path);
    printf("  \"vectors_tested\": %d,\n", n);
    printf("  \"threshold\": %.6f,\n", (double)HIF_THRESHOLD);
    printf("  \"persist_windows\": %d,\n", HIF_PERSIST);
    printf("  \"params\": %d,\n", params);
    printf("  \"model_bytes_int8_weights_float_bias\": %d,\n", model_bytes);
    printf("  \"max_abs_prob_error\": %.9e,\n", max_abs_err);
    printf("  \"mean_abs_prob_error\": %.9e,\n", mean_abs_err);
    printf("  \"decision_matches\": %d,\n", decision_matches);
    printf("  \"decision_agreement\": %.6f,\n", (double)decision_matches / (double)n);
    printf("  \"persistence_state_machine_pass\": %s,\n", (persist_ok == 1) ? "true" : "false");
    printf("  \"benchmark_iterations\": %d,\n", iters);
    printf("  \"host_latency_us_per_inference\": %.4f,\n", us_per_inf);
    printf("  \"parity_pass\": %s\n", parity_pass ? "true" : "false");
    printf("}\n");

    return parity_pass ? 0 : 1;
}


In [ ]:
import importlib, train_kaggle
importlib.reload(train_kaggle)
train_kaggle.main()

In [ ]:
import json; print(json.dumps(json.load(open('results/metrics.json')), indent=1)[:6000])

## Phase 1: Zero-shot evaluation on REAL VeHIF 22 kV recordings

In [ ]:
import importlib, real_data_test
importlib.reload(real_data_test)
real_data_test.main()

## Phase 1.5: GCC C firmware parity + 50 Hz VN vs 60 Hz US + feature ablation + few-shot VeHIF adaptation

In [ ]:
import importlib, phase15_vehif_adaptation
importlib.reload(phase15_vehif_adaptation)
phase15_vehif_adaptation.main()